# Assignment 02: Movie Recommender Systems

This notebook implements two recommendation approaches for the data mining assignment:

1. **Content-based recommendation** using IMDb metadata, CMU plot summaries, and Netflix movie/rating information.
2. **Collaborative filtering** using Netflix user-movie ratings with user-user and item-item similarity.

The notebook keeps dataset-preparation blocks for reproducibility. Some long preprocessing cells are commented because the cleaned intermediate files were saved and reused during experimentation.


## Environment and Path Configuration

Update the paths below before running the notebook on a new machine. The raw datasets are not included in the GitHub repository because they are large and must be downloaded from their original sources.


In [ ]:
from pathlib import Path

# Update this folder after downloading/preparing the datasets.
DATA_DIR = Path("data")
PROCESSED_DIR = DATA_DIR / "processed"
FINAL_DATASET_PATH = DATA_DIR / "imdb_cmu_netflix_verified.csv"
NETFLIX_RATINGS_PATH = DATA_DIR / "netflix_ratings_filtered.csv"
PROBE_RATINGS_PATH = DATA_DIR / "probe_filtered.csv"

# Generated artifacts
MODEL_DIR = Path("saved_models")
MODEL_DIR.mkdir(exist_ok=True)


## Library imports

In [ ]:
# Install once if needed:
# %pip install pandas numpy matplotlib seaborn pyarrow tqdm scikit-learn


## Dataset description

### IMDb data

In [ ]:
# import pandas as pd
# import os


# imdb_path = "D:\dataset_assignment2\dataset\imdb"


# files = [f for f in os.listdir(imdb_path) if f.endswith(".tsv")]

# for f in files:
#     file_path = os.path.join(imdb_path, f)
#     try:
       
#         df = pd.read_csv(file_path, sep="\t", dtype=str, nrows=5000)
#         total_rows = sum(1 for _ in open(file_path, encoding="utf-8", errors="ignore")) - 1  
#         print(f" {f}")
#         print(f"   Rows (approx): {total_rows:,}")
#         print(f"   Columns: {len(df.columns)}")
#         print(f"   Column names: {list(df.columns)}\n")
#     except Exception as e:
#         print(f" Could not read {f} → {e}\n")


### CMU plot summary data

In [ ]:
# import pandas as pd
# import os


# cmu_path = "\dataset_assignment2\dataset\moviesummary"


# files = [f for f in os.listdir(cmu_path) if f.endswith((".tsv", ".txt"))]


# for f in files:
#     file_path = os.path.join(cmu_path, f)
#     print(f" {f}")
#     try:
      
#         total_rows = sum(1 for _ in open(file_path, encoding="utf-8", errors="ignore"))
      
#         df = pd.read_csv(file_path, sep="\t", header=None, nrows=5000, dtype=str)
#         print(f"   Rows (approx): {total_rows:,}")
#         print(f"   Columns: {len(df.columns)}")
#         print(f"   Column names: {list(df.columns)}\n")
#     except Exception as e:
#         print(f"  Could not read this file ({e})\n")


### Netflix rating data

In [ ]:

# import pandas as pd
# import os


# netflix_path =r"dataset\netflix_data"


# files = os.listdir(netflix_path)

# for f in files:
#     file_path = os.path.join(netflix_path, f)
#     print(f" {f}")
#     try:
        
#         if f.endswith(".csv"):
#             df = pd.read_csv(file_path, encoding="latin-1", header=None)
#             print(f"   Rows: {df.shape[0]:,}")
#             print(f"   Columns: {df.shape[1]}")
#             print(f"   Column names: {list(df.columns)}\n")

       
#         elif f.endswith(".txt"):
#             with open(file_path, "r", encoding="latin-1", errors="ignore") as file:
#                 lines = [next(file) for _ in range(5)]  # read first 5 lines
#             total_rows = sum(1 for _ in open(file_path, encoding="latin-1", errors="ignore"))
#             print(f"   Rows (approx): {total_rows:,}")
#             print(f"   Columns: varies by structure (not fixed)")
#             print("   Sample lines:")
#             for l in lines:
#                 print("     ", l.strip())
#             print("\n")

#         else:
#             print("   Warning: Unsupported file type\n")
#     except Exception as e:
#         print(f"    Could not read this file → {e}\n")


## Part 1: Data preparation

• Load and clean both datasets.

• Handle missing values (e.g., missing directors or plot summaries).

• Merge IMDB and Netflix data by movie title or ID (if applicable).

• Create a unified dataframe with relevant content features and average ratings.

### 1.1 Preprocess IMDb metadata

In [ ]:
# from pathlib import Path
# import pandas as pd, numpy as np, re, datetime as dt

# CHUNK = 1_000_000

# def norm_title(t):
#     if pd.isna(t): return np.nan
#     t = t.lower().strip()
#     t = re.sub(r"[^\w\s]", " ", t)
#     t = re.sub(r"\s+", " ", t)
#     t = re.sub(r"^(the|a|an)\s+", "", t)
#     return t

# def header(path):
#     return list(pd.read_csv(path, sep="\t", nrows=0).columns)

# def write_both(df, outdir, base):
#     outdir = Path(outdir)
#     outdir.mkdir(parents=True, exist_ok=True)

#     # always write CSV
#     csv_path = outdir / f"{base}_slim.csv"
#     df.to_csv(csv_path, index=False, encoding="utf-8", lineterminator="\n")

#     # try parquet
#     parquet_path = outdir / f"{base}_slim.parquet"
#     try:
#         df.to_parquet(parquet_path, index=False)
#         print(f" Saved: {csv_path.name}, {parquet_path.name}")
#     except Exception as e:
#         print(f" Saved: {csv_path.name}")
#         print(f"Warning: Could not write parquet ({e}). Install 'pyarrow' or 'fastparquet' to enable parquet output.")

#     return csv_path, parquet_path

# def write_audit(outdir, base, kept_cols, dropped_cols_count, rows_dropped):
#     outdir = Path(outdir)
#     audit_path = outdir / f"{base}_slim.audit.txt"
#     with open(audit_path, "w", encoding="utf-8") as f:
#         f.write(f"IMDb FILE AUDIT\nGenerated: {dt.datetime.now().isoformat(timespec='seconds')}\n")
#         f.write(f"Kept columns: {list(kept_cols)}\n")
#         f.write(f"Dropped columns: {int(dropped_cols_count)}\n")
#         f.write(f"Rows dropped (NaN/empty per rules): {int(rows_dropped)}\n")
#     print(f"🧾 Audit: {audit_path.name}")

# def process_file(input_path, outdir):
#     path = Path(input_path)
#     name = path.name.lower()
#     hdr = header(path)

#     # --- BASICS ---
#     if "title.basics.tsv" in name:
#         usecols = ["tconst","titleType","primaryTitle","startYear","genres"]
#         dropped_cols = [c for c in hdr if c not in usecols]
#         df = pd.read_csv(path, sep="\t", na_values="\\N", dtype="string", usecols=usecols)
#         df = df[df["titleType"]=="movie"][["tconst","primaryTitle","startYear","genres"]]
#         df = df.rename(columns={"primaryTitle":"title","genres":"genre"})
#         df["year"] = pd.to_numeric(df["startYear"], errors="coerce")
#         df["title_norm"] = df["title"].apply(norm_title)
#         df["genre"] = df["genre"].fillna("").astype("string")
#         before = len(df)
#         df = df.dropna(subset=["tconst","title","year"])
#         dropped = before - len(df)
#         current_year = dt.datetime.now().year
#         before_year = len(df)
#         df = df[(df["year"] >= 1888) & (df["year"] <= current_year + 1)]
#         dropped += before_year - len(df)
#         df["year"] = df["year"].astype("Int16")
#         df = df[["tconst","title","year","genre","title_norm"]]
#         base = "basics"

#     # --- AKAS ---
#     elif "title.akas.tsv" in name:
#         usecols = ["titleId","language"]
#         dropped_cols = [c for c in hdr if c not in usecols]
#         df = pd.read_csv(path, sep="\t", na_values="\\N", dtype="string", usecols=usecols)
#         df = df.rename(columns={"titleId":"tconst"})
#         before = len(df)
#         df = df.dropna(subset=["tconst"])
#         dropped = before - len(df)
#         df["language"] = df["language"].fillna("").astype("string")
#         df = df[["tconst","language"]]
#         base = "akas"

#     # --- CREW ---
#     elif "title.crew.tsv" in name:
#         usecols = ["tconst","directors","writers"]
#         dropped_cols = [c for c in hdr if c not in usecols]
#         df = pd.read_csv(path, sep="\t", na_values="\\N", dtype="string", usecols=usecols)
#         before = len(df)
#         df = df.dropna(subset=["tconst"])
#         df[["directors","writers"]] = df[["directors","writers"]].fillna("").astype("string")
#         # drop rows where both are empty
#         empty_both = (df["directors"]== "") & (df["writers"]=="")
#         df = df[~empty_both]
#         dropped = before - len(df)
#         df = df[["tconst","directors","writers"]]
#         base = "crew"

#     # --- PRINCIPALS ---
#     elif "title.principals.tsv" in name:
#         usecols = ["tconst","nconst","category","ordering"]
#         dropped_cols = [c for c in hdr if c not in usecols]
#         parts, total_before = [], 0
#         for chunk in pd.read_csv(path, sep="\t", na_values="\\N", dtype="string",
#                                  usecols=usecols, chunksize=CHUNK):
#             total_before += len(chunk)
#             chunk["ordering"] = pd.to_numeric(chunk["ordering"], errors="coerce")
#             chunk = chunk[(chunk["category"].isin(["actor","actress"])) & (chunk["ordering"] <= 10)]
#             chunk = chunk.dropna(subset=["tconst","nconst"])[["tconst","nconst"]]
#             parts.append(chunk)
#         df = pd.concat(parts, ignore_index=True)
#         dropped = total_before - len(df)
#         base = "principals"

#     # --- NAMES ---
#     elif "name.basics.tsv" in name:
#         usecols = ["nconst","primaryName"]
#         dropped_cols = [c for c in hdr if c not in usecols]
#         parts, total_before = [], 0
#         for chunk in pd.read_csv(path, sep="\t", na_values="\\N", dtype="string",
#                                  usecols=usecols, chunksize=CHUNK):
#             total_before += len(chunk)
#             chunk["primaryName"] = chunk["primaryName"].fillna("").astype("string").str.strip()
#             chunk = chunk.dropna(subset=["nconst"])
#             chunk = chunk[chunk["primaryName"] != ""]
#             parts.append(chunk[["nconst","primaryName"]])
#         df = pd.concat(parts, ignore_index=True)
#         dropped = total_before - len(df)
#         base = "names"

#     else:
#         raise ValueError("Unsupported IMDb file name; expected one of the standard IMDb TSVs.")

#     # save
#     write_both(df, outdir, base)
#     write_audit(outdir, base, df.columns, len(dropped_cols), dropped)
#     return df


In [ ]:
# df_basics = process_file(
#     r"D:\dataset_assignment2\dataset\imdb\title.basics.tsv",
#     r"D:\dataset_assignment2\dataset\processed"
# )


In [ ]:
# df_basics = process_file(
#     r"D:\dataset_assignment2\dataset\imdb\name.basics.tsv",
#     r"D:\dataset_assignment2\dataset\processed\imdb"
# )


In [ ]:
# df_basics = process_file(
#     r"D:\dataset_assignment2\dataset\imdb\title.akas.tsv",
#     r"D:\dataset_assignment2\dataset\processed\imdb"
# )


In [ ]:
# df_basics = process_file(
#     r"D:\dataset_assignment2\dataset\imdb\title.crew.tsv",
#     r"D:\dataset_assignment2\dataset\processed\imdb"
# )


In [ ]:
# df_basics = process_file(
#     r"D:\dataset_assignment2\dataset\imdb\title.principals.tsv",
#     r"D:\dataset_assignment2\dataset\processed\imdb"
# )


### 1.2 Merge IMDb files into a single metadata table

In [ ]:
# # ===== IMDb MERGE — 
# from pathlib import Path
# from collections import defaultdict, Counter
# import pandas as pd
# import numpy as np
# import datetime as dt
# import gc
# import pyarrow.parquet as pq  


# in_dir  = Path(r"D:\dataset_assignment2\dataset\processed\imdb")
# out_dir = Path(r"D:\dataset_assignment2\dataset\final")
# out_dir.mkdir(parents=True, exist_ok=True)
# MAX_STARS = 20

# # ---------- Helper functions ----------
# def load(base):
#     pq_path = in_dir / f"{base}_slim.parquet"
#     csv_path = in_dir / f"{base}_slim.csv"
#     if pq_path.exists():
#         print(f" Loading {pq_path.name}")
#         return pd.read_parquet(pq_path)
#     print(f" Loading {csv_path.name}")
#     return pd.read_csv(csv_path, dtype=str)

# def ids_to_names(id_string: str, id2name: dict) -> str:
#     if not isinstance(id_string, str) or not id_string.strip():
#         return ""
#     out, seen = [], set()
#     for i in id_string.split(","):
#         nm = id2name.get(i, "")
#         if nm and nm not in seen:
#             seen.add(nm); out.append(nm)
#     return ", ".join(out)

# # ---------- Stage 0: Load core datasets ----------
# print(" Loading basics and names ...")
# basics = load("basics")
# names_df = load("names")

# wanted = set(basics["tconst"].astype(str))
# id2name = dict(zip(names_df["nconst"].astype(str), names_df["primaryName"].astype(str)))
# del names_df
# gc.collect()

# # # ----------------------------------------------------------------------
# # STAGE 1 — Build stars (principals)
# # # ----------------------------------------------------------------------
# part_stars_path = out_dir / "imdb_part_stars.parquet"
# if not part_stars_path.exists():
#     print(" Building stars from principals_slim (batched) ...")
#     p_princ = in_dir / "principals_slim.parquet"
#     tconst_to_names = defaultdict(set)

#     if p_princ.exists():
#         pfile = pq.ParquetFile(p_princ)
#         for rg in range(pfile.num_row_groups):
#             batch = pfile.read_row_group(rg, columns=["tconst","nconst"]).to_pandas()
#             batch["tconst"] = batch["tconst"].astype(str)
#             batch = batch[batch["tconst"].isin(wanted)]
#             if batch.empty:
#                 continue
#             batch["name"] = batch["nconst"].astype(str).map(id2name).fillna("")
#             batch = batch[batch["name"] != ""]
#             grouped = batch.groupby("tconst", observed=True)["name"].apply(lambda s: set(s.unique()))
#             for t, sset in grouped.items():
#                 tconst_to_names[t] |= sset
#     else:
#         CHUNK = 1_000_000
#         for chunk in pd.read_csv(in_dir / "principals_slim.csv", dtype=str,
#                                  usecols=["tconst","nconst"], chunksize=CHUNK):
#             chunk["tconst"] = chunk["tconst"].astype(str)
#             chunk = chunk[chunk["tconst"].isin(wanted)]
#             if chunk.empty:
#                 continue
#             chunk["name"] = chunk["nconst"].astype(str).map(id2name).fillna("")
#             chunk = chunk[chunk["name"] != ""]
#             grouped = chunk.groupby("tconst", observed=True)["name"].apply(lambda s: set(s.unique()))
#             for t, sset in grouped.items():
#                 tconst_to_names[t] |= sset

#     stars_df = pd.DataFrame({
#         "tconst": list(tconst_to_names.keys()),
#         "stars":  [", ".join(sorted(list(s))[:MAX_STARS]) for s in tconst_to_names.values()]
#     })
#     stars_df.to_parquet(part_stars_path, index=False)
#     print(" Stars built & saved")
# else:
#     print(" Stars already built — loading from file")
#     stars_df = pd.read_parquet(part_stars_path)

# gc.collect()

# # # ----------------------------------------------------------------------
# # STAGE 2 — Build language (akas)
# # # ----------------------------------------------------------------------
# part_lang_path = out_dir / "imdb_part_lang.parquet"
# if not part_lang_path.exists():
#     print(" Deriving dominant language from akas (batched) ...")
#     lang_counts = {}
#     p_akas = in_dir / "akas_slim.parquet"

#     if p_akas.exists():
#         afile = pq.ParquetFile(p_akas)
#         for rg in range(afile.num_row_groups):
#             batch = afile.read_row_group(rg, columns=["tconst","language"]).to_pandas()
#             batch["tconst"] = batch["tconst"].astype(str)
#             batch = batch[batch["tconst"].isin(wanted)]
#             if batch.empty:
#                 continue
#             batch["language"] = batch["language"].fillna("")
#             for t, s in batch.groupby("tconst", observed=True)["language"]:
#                 if t not in lang_counts:
#                     lang_counts[t] = Counter()
#                 lang_counts[t].update(s[s != ""].tolist())
#     else:
#         CHUNK = 1_000_000
#         for chunk in pd.read_csv(in_dir / "akas_slim.csv", dtype=str,
#                                  usecols=["tconst","language"], chunksize=CHUNK):
#             chunk["tconst"] = chunk["tconst"].astype(str)
#             chunk = chunk[chunk["tconst"].isin(wanted)]
#             if chunk.empty:
#                 continue
#             chunk["language"] = chunk["language"].fillna("")
#             for t, s in chunk.groupby("tconst", observed=True)["language"]:
#                 if t not in lang_counts:
#                     lang_counts[t] = Counter()
#                 lang_counts[t].update(s[s != ""].tolist())

#     lang = pd.DataFrame({
#         "tconst": list(lang_counts.keys()),
#         "language": [(cnt.most_common(1)[0][0] if cnt else "") for cnt in lang_counts.values()]
#     })
#     lang.to_parquet(part_lang_path, index=False)
#     print(" Language ready & saved")
# else:
#     print(" Language already built — loading from file")
#     lang = pd.read_parquet(part_lang_path)

# gc.collect()

# # # ----------------------------------------------------------------------
# # STAGE 3 — Build crew (directors/writers)
# # # ----------------------------------------------------------------------
# part_crew_path = out_dir / "imdb_part_crew.parquet"
# if not part_crew_path.exists():
#     print(" Expanding crew IDs → names (batched + filtered) ...")
#     p_crew = in_dir / "crew_slim.parquet"
#     crew_rows = []

#     if p_crew.exists():
#         cfile = pq.ParquetFile(p_crew)
#         for rg in range(cfile.num_row_groups):
#             batch = cfile.read_row_group(rg, columns=["tconst","directors","writers"]).to_pandas()
#             batch["tconst"] = batch["tconst"].astype(str)
#             batch = batch[batch["tconst"].isin(wanted)]
#             if batch.empty:
#                 continue
#             crew_rows.append(batch)
#     else:
#         CHUNK = 1_000_000
#         for chunk in pd.read_csv(in_dir / "crew_slim.csv", dtype=str,
#                                  usecols=["tconst","directors","writers"], chunksize=CHUNK):
#             chunk["tconst"] = chunk["tconst"].astype(str)
#             chunk = chunk[chunk["tconst"].isin(wanted)]
#             if chunk.empty:
#                 continue
#             crew_rows.append(chunk)

#     crew_df = pd.concat(crew_rows, ignore_index=True)
#     crew_df["directors"] = crew_df["directors"].astype(str).map(lambda s: ids_to_names(s, id2name))
#     crew_df["writers"]   = crew_df["writers"].astype(str).map(lambda s: ids_to_names(s, id2name))
#     crew_df.to_parquet(part_crew_path, index=False)
#     print(" Crew expanded & saved")
# else:
#     print(" Crew already built — loading from file")
#     crew_df = pd.read_parquet(part_crew_path)

# gc.collect()

# # # ----------------------------------------------------------------------
# # STAGE 4 — Merge incrementally
# # # ----------------------------------------------------------------------
# print(" Merging basics + language ...")
# imdb_part1 = basics.merge(lang, on="tconst", how="left")
# imdb_part1.to_parquet(out_dir / "imdb_part1_basics_lang.parquet", index=False)
# print(" basics + language merged & saved")

# del basics, lang
# gc.collect()

# print(" Loading part1 and merging with crew ...")
# imdb_part1 = pd.read_parquet(out_dir / "imdb_part1_basics_lang.parquet")
# imdb_part2 = imdb_part1.merge(crew_df, on="tconst", how="left")
# imdb_part2.to_parquet(out_dir / "imdb_part2_add_crew.parquet", index=False)
# print(" crew merged & saved")

# del imdb_part1, crew_df
# gc.collect()

# print(" Loading part2 and merging with stars ...")
# imdb_part2 = pd.read_parquet(out_dir / "imdb_part2_add_crew.parquet")
# imdb_ready = imdb_part2.merge(stars_df, on="tconst", how="left")
# print(" stars merged")

# del imdb_part2, stars_df
# gc.collect()

# # # ----------------------------------------------------------------------
# # STAGE 5 — Final cleanup and save
# # # ----------------------------------------------------------------------
# current_year = dt.datetime.now().year
# for col in ["title","genre","language","directors","writers","stars","title_norm"]:
#     imdb_ready[col] = imdb_ready[col].fillna("").astype("string")

# imdb_ready = imdb_ready.drop_duplicates(subset=["tconst"]).reset_index(drop=True)
# imdb_ready = imdb_ready[(imdb_ready["year"].astype("Int32") >= 1888) &
#                         (imdb_ready["year"].astype("Int32") <= current_year + 1)]
# imdb_ready["year"] = imdb_ready["year"].astype("Int16")

# final_cols = ["tconst","title","year","genre","language","directors","writers","stars","title_norm"]
# imdb_ready = imdb_ready.reindex(columns=final_cols)

# # Save final dataset
# csv_path = out_dir / "imdb_ready.csv"
# parquet_path = out_dir / "imdb_ready.parquet"
# audit_path = out_dir / "imdb_ready.audit.txt"

# imdb_ready.to_parquet(parquet_path, index=False)
# imdb_ready.to_csv(csv_path, index=False, encoding="utf-8", lineterminator="\n")

# with open(audit_path, "w", encoding="utf-8") as f:
#     f.write("IMDb READY — AUDIT\n")
#     f.write(f"Generated: {dt.datetime.now().isoformat(timespec='seconds')}\n")
#     f.write(f"Columns: {final_cols}\n")
#     f.write(f"Shape: {imdb_ready.shape[0]} rows × {imdb_ready.shape[1]} cols\n")

# print(" IMDb dataset fully ready and saved")


### 1.3 Preprocess CMU movie summary data

In [ ]:

# from pathlib import Path
# import pandas as pd, numpy as np, re, datetime as dt

# # --- EDIT PATHS ---
# cmu_dir = Path(r"D:\dataset_assignment2\dataset\moviesummary")
# out_dir = Path(r"D:\dataset_assignment2\dataset\final\cmu")
# out_dir.mkdir(parents=True, exist_ok=True)

# meta_path  = cmu_dir / "movie.metadata.tsv"
# out_parq   = out_dir / "cmu_meta_ready.parquet"
# out_csv    = out_dir / "cmu_meta_ready.csv"
# out_audit  = out_dir / "cmu_meta_ready.audit.txt"

# # --- helpers ---
# def norm_title(t: str):
#     if t is None or (isinstance(t, float) and np.isnan(t)): return np.nan
#     t = t.lower().strip()
#     t = re.sub(r"[^\w\s]", " ", t)
#     t = re.sub(r"\s+", " ", t)
#     t = re.sub(r"^(the|a|an)\s+", "", t)
#     return t

# def year_from_date(s: str):
#     if not isinstance(s, str): return np.nan
#     m = re.search(r"\b(18|19|20)\d{2}\b", s)
#     return int(m.group(0)) if m else np.nan

# def parse_fb_field(s: str) -> str:
#     """Extract human-readable names from 'FreebaseID:name' tuples."""
#     if not isinstance(s, str) or not s.strip(): return ""
#     t = re.sub(r"[\[\]\{\}\(\)]", " ", s)
#     candidates = [p for p in re.split(r"[|,]", t) if ":" in p]
#     names = []
#     seen = set()
#     for p in candidates:
#         name = p.split(":", 1)[1].strip()
#         if name and name not in seen:
#             seen.add(name)
#             names.append(name)
#     return ", ".join(names)

# print(" Reading movie.metadata.tsv (needed columns only) ...")
# cols_all = ["wiki_movie_id","freebase_id","movie_name","release_date","box_office","runtime","languages","countries","genres"]
# use = ["wiki_movie_id","movie_name","release_date","languages","genres"]

# meta = pd.read_csv(
#     meta_path, sep="\t", header=None, names=cols_all,
#     usecols=use, dtype=str, low_memory=False
# )

# print(" Cleaning metadata ...")
# meta["release_year"] = meta["release_date"].apply(year_from_date)
# meta["languages"]    = meta["languages"].apply(parse_fb_field)
# meta["genres"]       = meta["genres"].apply(parse_fb_field)
# meta["movie_name"]   = meta["movie_name"].fillna("").str.strip()
# meta["title_norm"]   = meta["movie_name"].apply(norm_title)

# before = len(meta)
# meta = meta.dropna(subset=["movie_name","release_year"])
# after  = len(meta)

# meta = meta[["wiki_movie_id","movie_name","release_year","languages","genres","title_norm"]].copy()
# meta["wiki_movie_id"] = meta["wiki_movie_id"].astype(str)

# print(" Saving cmu_meta_ready ...")
# meta.to_csv(out_csv, index=False, encoding="utf-8", lineterminator="\n")
# try:
#     meta.to_parquet(out_parq, index=False)
# except Exception as e:
#     print(f"Warning: Parquet skipped: {e}")

# with open(out_audit, "w", encoding="utf-8") as f:
#     f.write("CMU META READY — AUDIT\n")
#     f.write(f"Generated: {dt.datetime.now().isoformat(timespec='seconds')}\n")
#     f.write("Kept columns: ['wiki_movie_id','movie_name','release_year','languages','genres','title_norm']\n")
#     f.write(f"Rows dropped for missing criticals (title/year): {before - after}\n")
#     f.write(f"Final shape: {meta.shape[0]} rows × {meta.shape[1]} cols\n")

# print(" cmu_meta_ready saved")


In [ ]:

# from pathlib import Path
# import pandas as pd, numpy as np, datetime as dt

# # --- EDIT PATHS ---
# cmu_dir = Path(r"D:\dataset_assignment2\dataset\moviesummary")
# out_dir = Path(r"D:\dataset_assignment2\dataset\final\cmu")
# out_dir.mkdir(parents=True, exist_ok=True)

# plots_path = cmu_dir / "plot_summaries.txt"
# out_parq   = out_dir / "cmu_plots_ready.parquet"
# out_csv    = out_dir / "cmu_plots_ready.csv"
# out_audit  = out_dir / "cmu_plots_ready.audit.txt"

# print(" Reading plot_summaries.txt in chunks ...")
# CHUNK = 400_000
# parts = []
# total_before = 0
# total_after  = 0

# for chunk in pd.read_csv(
#     plots_path, sep="\t", header=None, names=["wiki_movie_id","plot_summary"],
#     dtype=str, chunksize=CHUNK, quoting=3, encoding_errors="ignore"
# ):
#     total_before += len(chunk)
#     chunk["wiki_movie_id"] = chunk["wiki_movie_id"].astype(str)
#     chunk["plot_summary"]  = chunk["plot_summary"].fillna("").str.strip()
#     chunk = chunk[chunk["plot_summary"] != ""]
#     total_after += len(chunk)
#     parts.append(chunk)

# plots = pd.concat(parts, ignore_index=True)
# print(" Saving cmu_plots_ready ...")
# plots.to_csv(out_csv, index=False, encoding="utf-8", lineterminator="\n")
# try:
#     plots.to_parquet(out_parq, index=False)
# except Exception as e:
#     print(f"Warning: Parquet skipped: {e}")

# with open(out_audit, "w", encoding="utf-8") as f:
#     f.write("CMU PLOTS READY — AUDIT\n")
#     f.write(f"Generated: {dt.datetime.now().isoformat(timespec='seconds')}\n")
#     f.write("Kept columns: ['wiki_movie_id','plot_summary']\n")
#     f.write(f"Dropped empty summaries: {total_before - total_after}\n")
#     f.write(f"Final shape: {plots.shape[0]} rows × {plots.shape[1]} cols\n")

# print(" cmu_plots_ready saved")


In [ ]:
# # ==== CMU STEP 3 — Merge cmu_meta_ready + cmu_plots_ready → cmu_ready ====
# from pathlib import Path
# import pandas as pd, numpy as np, datetime as dt

# # --- EDIT PATHS ---
# final_dir = Path(r"D:\dataset_assignment2\dataset\final\cmu")
# final_dir.mkdir(parents=True, exist_ok=True)

# meta_parq  = final_dir / "cmu_meta_ready.parquet"
# plots_parq = final_dir / "cmu_plots_ready.parquet"
# meta_csv   = final_dir / "cmu_meta_ready.csv"
# plots_csv  = final_dir / "cmu_plots_ready.csv"

# out_parq   = final_dir / "cmu_ready.parquet"
# out_csv    = final_dir / "cmu_ready.csv"
# out_audit  = final_dir / "cmu_ready.audit.txt"

# # ---- Load (prefer Parquet) ----
# print(" Loading cmu_meta_ready ...")
# meta  = pd.read_parquet(meta_parq)  if meta_parq.exists()  else pd.read_csv(meta_csv, dtype=str)
# print(" Loading cmu_plots_ready ...")
# plots = pd.read_parquet(plots_parq) if plots_parq.exists() else pd.read_csv(plots_csv, dtype=str)

# # Ensure types/columns are as expected
# meta["wiki_movie_id"]  = meta["wiki_movie_id"].astype(str)
# plots["wiki_movie_id"] = plots["wiki_movie_id"].astype(str)
# if "plot_summary" not in plots.columns:
#     plots.rename(columns={plots.columns[-1]: "plot_summary"}, inplace=True)

# # ---- Merge (left) ----
# print(" Merging metadata + plots ...")
# cmu_ready = meta.merge(plots, on="wiki_movie_id", how="left")

# # Minimal cleanup
# cmu_ready["plot_summary"] = cmu_ready["plot_summary"].fillna("").astype("string")

# # ---- Save ----
# print(" Saving cmu_ready ...")
# cmu_ready.to_csv(out_csv, index=False, encoding="utf-8", lineterminator="\n")
# try:
#     cmu_ready.to_parquet(out_parq, index=False)
# except Exception as e:
#     print(f"Warning: Parquet skipped: {e}")

# # ---- Audit ----
# matched_plots = int((cmu_ready["plot_summary"] != "").sum())
# with open(out_audit, "w", encoding="utf-8") as f:
#     f.write("CMU READY — AUDIT\n")
#     f.write(f"Generated: {dt.datetime.now().isoformat(timespec='seconds')}\n")
#     f.write("Columns: ['wiki_movie_id','movie_name','release_year','languages','genres','plot_summary','title_norm']\n")
#     f.write(f"Rows (meta): {len(meta)}\n")
#     f.write(f"Rows (plots): {len(plots)}\n")
#     f.write(f"Rows (merged): {len(cmu_ready)}\n")
#     f.write(f"Plot summaries matched (non-empty): {matched_plots}\n")

# print(" cmu_ready saved")


### 1.4 Merge CMU plot summaries with IMDb metadata

In [ ]:
# # ==== IMDb + CMU MERGE — exact + fallback (closest year) ====
# from pathlib import Path
# import pandas as pd, numpy as np, datetime as dt

# # --- PATHS ---
# imdb_dir = Path(r"D:\dataset_assignment2\dataset\final\imdb")    # IMDb input folder
# cmu_dir  = Path(r"D:\dataset_assignment2\dataset\final\cmu")     # CMU input folder
# out_dir  = Path(r"D:\dataset_assignment2\dataset\merged")        # output folder
# out_dir.mkdir(parents=True, exist_ok=True)

# imdb_path_parq = imdb_dir / "imdb_ready.parquet"
# imdb_path_csv  = imdb_dir / "imdb_ready.csv"
# cmu_path_parq  = cmu_dir / "cmu_ready.parquet"
# cmu_path_csv   = cmu_dir / "cmu_ready.csv"

# out_parq  = out_dir / "imdb_cmu_ready.parquet"
# out_csv   = out_dir / "imdb_cmu_ready.csv"
# out_audit = out_dir / "imdb_cmu_ready.audit.txt"

# # --- Load (prefer Parquet) ---
# print(" Loading IMDb & CMU ...")
# imdb = pd.read_parquet(imdb_path_parq) if imdb_path_parq.exists() else pd.read_csv(imdb_path_csv, dtype=str)
# cmu  = pd.read_parquet(cmu_path_parq)  if cmu_path_parq.exists()  else pd.read_csv(cmu_path_csv, dtype=str)

# # --- Types & minimal prep ---
# imdb["title_norm"]  = imdb["title_norm"].astype("string").fillna("")
# cmu["title_norm"]   = cmu["title_norm"].astype("string").fillna("")
# imdb["year"]        = pd.to_numeric(imdb["year"], errors="coerce").astype("Int32")
# cmu["release_year"] = pd.to_numeric(cmu["release_year"], errors="coerce").astype("Int32")

# cmu_keep = cmu[["title_norm","release_year","plot_summary","languages","genres","wiki_movie_id"]].copy()
# cmu_keep = cmu_keep.rename(columns={"languages":"cmu_languages","genres":"cmu_genres"})

# total_imdb = len(imdb)
# total_cmu  = len(cmu_keep)

# # # ----------------------------------------------------------------------
# # Stage 1: Exact match (title_norm + year)
# # # ----------------------------------------------------------------------
# print(" Exact merge on title_norm + year ...")
# m1 = imdb.merge(
#     cmu_keep,
#     left_on=["title_norm","year"],
#     right_on=["title_norm","release_year"],
#     how="left",
#     suffixes=("", "_cmu")
# )
# exact_matches = int(m1["plot_summary"].notna().sum())

# # # ----------------------------------------------------------------------
# # Stage 2: Fallback (title_norm only, pick closest year)
# # # ----------------------------------------------------------------------
# print(" Fallback merge on title_norm (closest year) ...")
# unmatched_mask = m1["plot_summary"].isna()
# imdb_un = m1.loc[unmatched_mask, ["tconst","title_norm","year"]].copy()

# if not imdb_un.empty:
#     cand = imdb_un.merge(cmu_keep, on="title_norm", how="left")
#     cand["year"]         = pd.to_numeric(cand["year"], errors="coerce")
#     cand["release_year"] = pd.to_numeric(cand["release_year"], errors="coerce")
#     cand["year_diff"]    = (cand["year"] - cand["release_year"]).abs().fillna(10**9)

#     picked_idx = cand.groupby("tconst", observed=True)["year_diff"].idxmin()
#     picked = cand.loc[picked_idx, ["tconst","plot_summary","wiki_movie_id","cmu_languages","cmu_genres","release_year"]]

#     fallback_cols = ["plot_summary","wiki_movie_id","cmu_languages","cmu_genres","release_year"]
#     merged_data = m1.loc[unmatched_mask, ["tconst"]].merge(picked, on="tconst", how="left")
#     m1.loc[unmatched_mask, fallback_cols] = merged_data[fallback_cols].to_numpy()

# fallback_matches = int(m1["plot_summary"].notna().sum()) - exact_matches

# # # ----------------------------------------------------------------------
# # Finalize columns
# # # ----------------------------------------------------------------------
# m1["language"] = m1["language"].fillna("").astype("string")
# # ---- FIX: use pandas mask instead of numpy where, then cast to pandas string ----
# m1["language"] = m1["language"].mask(m1["language"] == "", m1["cmu_languages"].fillna(""))
# m1["language"] = m1["language"].astype("string")

# final_cols = [
#     "tconst","title","year","genre","language","directors","writers","stars","title_norm",
#     "plot_summary","cmu_genres","cmu_languages","wiki_movie_id"
# ]
# imdb_cmu_ready = m1[final_cols].copy()
# imdb_cmu_ready["year"] = imdb_cmu_ready["year"].astype("Int16")
# imdb_cmu_ready = imdb_cmu_ready.drop_duplicates(subset=["tconst"]).reset_index(drop=True)

# # # ----------------------------------------------------------------------
# # Save + audit
# # # ----------------------------------------------------------------------
# print(" Saving imdb_cmu_ready ...")
# imdb_cmu_ready.to_csv(out_csv, index=False, encoding="utf-8", lineterminator="\n")
# try:
#     imdb_cmu_ready.to_parquet(out_parq, index=False)
# except Exception as e:
#     print(f"Warning: Parquet skipped: {e}")

# matched_total = int(imdb_cmu_ready["plot_summary"].ne("").sum())
# with open(out_audit, "w", encoding="utf-8") as f:
#     f.write("IMDb + CMU — AUDIT\n")
#     f.write(f"Generated: {dt.datetime.now().isoformat(timespec='seconds')}\n\n")
#     f.write(f"IMDb rows: {total_imdb}\n")
#     f.write(f"CMU rows: {total_cmu}\n")
#     f.write(f"Exact matches (title_norm+year): {exact_matches}\n")
#     f.write(f"Fallback matches (title_norm, closest year): {fallback_matches}\n")
#     f.write(f"Total with plot_summary: {matched_total}\n")
#     f.write(f"Final shape: {imdb_cmu_ready.shape[0]} rows × {imdb_cmu_ready.shape[1]} cols\n")
#     f.write("Columns: " + str(final_cols) + "\n")

# print(" IMDb + CMU merged and saved")


In [ ]:
# #  Load the merged dataset
# import pandas as pd
# from pathlib import Path

# merged_path = Path(r"D:\dataset_assignment2\dataset\merged\imdb_cmu_ready.parquet")

# if merged_path.exists():
#     df = pd.read_parquet(merged_path)
# else:
#     df = pd.read_csv(merged_path.with_suffix(".csv"))

# #  Show overall shape
# print(f"IMDb–CMU merged shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

# #  Peek at key columns
# cols_to_check = [
#     "title", "year", "genre", "cmu_genres",
#     "language", "cmu_languages",
#     "directors", "writers", "stars",
#     "plot_summary"
# ]
# cols_present = [c for c in cols_to_check if c in df.columns]

# print("\n--- Sample merged rows (side-by-side genre/lang check) ---\n")
# display(df[cols_present].sample(10, random_state=42))

# #  Optional: check how many rows have plot summaries (i.e., matched movies)
# matched = df["plot_summary"].notna().sum() if "plot_summary" in df.columns else 0
# print(f"\nMovies with plot summaries: {matched:,} ({matched/len(df)*100:.2f}%)")

# #  Optional: compare IMDb vs CMU genres/languages for a few cases
# print("\n--- Examples where IMDb genre is missing but CMU filled ---\n")
# display(df[(df["genre"].isna()) & (df["cmu_genres"].notna())][["title","year","genre","cmu_genres"]].head(10))

# print("\n--- Examples where IMDb language missing but CMU filled ---\n")
# display(df[(df["language"].isna()) & (df["cmu_languages"].notna())][["title","year","language","cmu_languages"]].head(10))


In [ ]:
# # ==== IMDb+CMU: unify genre/language, drop helpers, save clean + model-ready ====
# from pathlib import Path
# import pandas as pd, numpy as np, datetime as dt

# # --- Paths ---
# in_path_parq = Path(r"D:\dataset_assignment2\dataset\merged\imdb_cmu_ready.parquet")
# in_path_csv  = Path(r"D:\dataset_assignment2\dataset\merged\imdb_cmu_ready.csv")
# out_dir      = Path(r"D:\dataset_assignment2\dataset\merged")
# out_dir.mkdir(parents=True, exist_ok=True)

# clean_parq   = out_dir / "imdb_cmu_clean.parquet"        # keeps IDs for traceability
# clean_csv    = out_dir / "imdb_cmu_clean.csv"
# model_parq   = out_dir / "imdb_cmu_model_ready.parquet"  # only assignment-required columns
# model_csv    = out_dir / "imdb_cmu_model_ready.csv"
# audit_txt    = out_dir / "imdb_cmu_postmerge.audit.txt"

# # --- Load (prefer parquet) ---
# if in_path_parq.exists():
#     df = pd.read_parquet(in_path_parq)
# else:
#     df = pd.read_csv(in_path_csv, dtype=str)

# # Ensure columns exist
# for c in ["genre","cmu_genres","language","cmu_languages"]:
#     if c not in df.columns:
#         df[c] = ""

# # Normalize empties
# for c in ["genre","cmu_genres","language","cmu_languages"]:
#     df[c] = df[c].fillna("").astype(str).str.strip()

# # === Unify GENRE: prefer IMDb; fallback to CMU if IMDb empty ===
# # (IMDb empty means "", NaN already converted)
# genre_before_empty = int((df["genre"] == "").sum())
# df["genre"] = df["genre"].where(df["genre"] != "", df["cmu_genres"])
# genre_after_empty  = int((df["genre"] == "").sum())

# # === Unify LANGUAGE: prefer IMDb; fallback to CMU if IMDb empty ===
# lang_before_empty = int((df["language"] == "").sum())
# df["language"] = df["language"].where(df["language"] != "", df["cmu_languages"])
# lang_after_empty  = int((df["language"] == "").sum())

# # Drop CMU duplicates & helpers not needed further
# helpers_to_drop = [c for c in ["cmu_genres","cmu_languages","title_norm"] if c in df.columns]
# df_clean = df.drop(columns=helpers_to_drop)

# # --- Save "clean" (keeps IDs for traceability) ---
# df_clean.to_csv(clean_csv, index=False, encoding="utf-8", lineterminator="\n")
# try:
#     df_clean.to_parquet(clean_parq, index=False)
# except Exception as e:
#     print(f"Warning: Parquet (clean) skipped: {e}")

# # --- Build "model-ready" (drop IDs unless you want to keep them) ---
# # For assignment modeling you only need these:
# model_cols = ["title","year","genre","language","directors","writers","stars","plot_summary"]
# # If any are missing, create empty to avoid KeyErrors
# for c in model_cols:
#     if c not in df_clean.columns:
#         df_clean[c] = "" if c != "year" else np.nan

# df_model = df_clean[model_cols].copy()

# # Tidy types
# df_model["year"] = pd.to_numeric(df_model["year"], errors="coerce").astype("Int64")

# # Save model-ready
# df_model.to_csv(model_csv, index=False, encoding="utf-8", lineterminator="\n")
# try:
#     df_model.to_parquet(model_parq, index=False)
# except Exception as e:
#     print(f"Warning: Parquet (model) skipped: {e}")

# # --- Minimal audit ---
# with open(audit_txt, "w", encoding="utf-8") as f:
#     f.write("IMDb+CMU POST-MERGE — AUDIT\n")
#     f.write(f"Generated: {dt.datetime.now().isoformat(timespec='seconds')}\n\n")
#     f.write(f"Input shape: {df.shape[0]} rows × {df.shape[1]} cols\n")
#     f.write(f"Unified genre: emptied-before={genre_before_empty}, emptied-after={genre_after_empty}\n")
#     f.write(f"Unified language: emptied-before={lang_before_empty}, emptied-after={lang_after_empty}\n")
#     f.write(f"Dropped helper columns: {helpers_to_drop}\n")
#     f.write(f"Saved CLEAN (traceable): {clean_csv.name}, {clean_parq.name}\n")
#     f.write(f"Saved MODEL-READY: {model_csv.name}, {model_parq.name}\n")

# print(" Done. Wrote:")
# print(f"  - Clean (IDs kept): {clean_csv.name} / {clean_parq.name}")
# print(f"  - Model-ready cols: {model_csv.name} / {model_parq.name}")
# print(f"  - Audit: {audit_txt.name}")


In [ ]:
# import pandas as pd
# df = pd.read_parquet(r"D:\dataset_assignment2\dataset\merged\imdb_cmu_model_ready.parquet")
# df.columns


### 1.5 Preprocess Netflix movie and rating data

In [ ]:
# # === Netflix Prize: movie_titles → merge-ready (Parquet + CSV + audit) ===
# # Robust to unquoted commas in title; no ParserError.
# # Output: CSV, Parquet (if pyarrow/fastparquet installed), and an audit .txt

# from pathlib import Path
# import pandas as pd
# import re
# import datetime as dt

# # -------------------- EDIT THESE PATHS --------------------
# in_path  = Path(r"D:\dataset_assignment2\dataset\netflix_data\movie_titles.csv")  # or .txt
# out_dir  = Path(r"D:\dataset_assignment2\dataset\processed\netflix_prize")
# # ----------------------------------------------------------

# out_dir.mkdir(parents=True, exist_ok=True)
# out_parq = out_dir / "netflix_titles_ready.parquet"
# out_csv  = out_dir / "netflix_titles_ready.csv"
# out_aud  = out_dir / "netflix_titles_ready.audit.txt"


# def norm_title(t: str) -> str:
#     """Normalize title for joining:
#        - strip quotes/whitespace
#        - lowercase
#        - remove punctuation
#        - collapse spaces
#        - drop leading articles (the/a/an)"""
#     if not isinstance(t, str):
#         return ""
#     t = t.strip().strip('"').strip("'")
#     t = t.lower()
#     t = re.sub(r"[^\w\s]", " ", t)    # remove punctuation
#     t = re.sub(r"\s+", " ", t)        # collapse whitespace
#     t = re.sub(r"^(the|a|an)\s+", "", t)  # drop leading article
#     return t.strip()


# def read_movie_titles(path: Path) -> pd.DataFrame:
#     """Read Netflix Prize movie_titles using a bulletproof parser:
#        split each line on the first two commas only => [MovieID, year, title...]"""
#     encodings = ["latin-1", "utf-8-sig"]
#     last_err = None
#     for enc in encodings:
#         try:
#             rows = []
#             with open(path, "r", encoding=enc, errors="strict") as f:
#                 for ln, line in enumerate(f, start=1):
#                     line = line.rstrip("\r\n")
#                     if not line:
#                         continue
#                     parts = line.split(",", 2)
#                     if len(parts) < 3:
#                         # malformed line; skip but you could log if needed
#                         continue
#                     movie_id, year, title = parts[0].strip(), parts[1].strip(), parts[2].strip()
#                     rows.append((movie_id, year, title))
#             return pd.DataFrame(rows, columns=["MovieID", "year", "title"])
#         except UnicodeDecodeError as e:
#             last_err = e
#             continue
#     raise last_err if last_err else RuntimeError("Unable to read file with provided encodings")


# def main():
#     # ---------- Read ----------
#     raw = read_movie_titles(in_path)

#     # ---------- Coerce types & basic cleaning ----------
#     before = len(raw)

#     raw["title"] = raw["title"].astype(str).str.strip().str.strip('"').str.strip("'")

#     df = pd.DataFrame({
#         "MovieID": pd.to_numeric(raw["MovieID"], errors="coerce").astype("Int32"),
#         "year":    pd.to_numeric(raw["year"], errors="coerce").astype("Int32"),
#         "title":   raw["title"],
#     })

#     # Drop unusable rows
#     df = df.dropna(subset=["MovieID", "year"])
#     df = df[df["title"].str.len() > 0]
#     after_drop = len(df)

#     # ---------- Normalized title & de-dup ----------
#     df["title_norm"] = df["title"].apply(norm_title)

#     # dedupe on (title_norm, year); keep smallest MovieID deterministically
#     dedup_before = len(df)
#     df = (
#         df.sort_values(["title_norm", "year", "MovieID"])
#           .drop_duplicates(subset=["title_norm", "year"], keep="first")
#           .reset_index(drop=True)
#     )
#     dedup_after = len(df)

#     # ---------- Save ----------
#     # CSV always
#     df.to_csv(out_csv, index=False, encoding="utf-8", lineterminator="\n")

#     # Parquet if engine installed
#     try:
#         df.to_parquet(out_parq, index=False)
#         parquet_status = out_parq.name
#     except Exception as e:
#         parquet_status = f"(parquet skipped) — install 'pyarrow' or 'fastparquet' — {e}"

#     # ---------- Minimal audit ----------
#     with open(out_aud, "w", encoding="utf-8") as f:
#         f.write("NETFLIX PRIZE — TITLES READY (for merging)\n")
#         f.write(f"Generated: {dt.datetime.now().isoformat(timespec='seconds')}\n\n")
#         f.write(f"Input file: {in_path}\n")
#         f.write(f"Saved rows: {len(df)}\n")
#         f.write(f"Dropped missing/empty (MovieID/year/title): {before - after_drop}\n")
#         f.write(f"Deduplicated (title_norm+year): {dedup_before - dedup_after}\n")
#         f.write(f"Columns: {list(df.columns)}\n")

#     # ---------- Console summary ----------
#     print(" Saved:")
#     print("  -", out_csv.name)
#     print("  -", parquet_status)
#     print("  -", out_aud.name)


# if __name__ == "__main__":
#     main()


### 1.6 Merge IMDb, CMU, and Netflix data

In [ ]:
# # === Merge IMDb+CMU with Netflix Prize titles (Assignment 2) ===
# from pathlib import Path
# import pandas as pd
# import datetime as dt

# # --------- EDIT THESE FILE PATHS (inputs are files, not directories) ----------
# imdb_cmu_path = Path(r"D:\dataset_assignment2\dataset\merged\imdb_cmu_ready.parquet")
# netflix_path  = Path(r"D:\dataset_assignment2\dataset\processed\netflix_prize\netflix_titles_ready.parquet")
# out_dir       = Path(r"D:\dataset_assignment2\dataset\main_dataset")
# # ------------------------------------------------------------------------------

# out_dir.mkdir(parents=True, exist_ok=True)

# out_csv  = out_dir / "merged_imdb_cmu_netflix.csv"
# out_parq = out_dir / "merged_imdb_cmu_netflix.parquet"
# out_aud  = out_dir / "merged_imdb_cmu_netflix.audit.txt"

# def smart_read(path: Path) -> pd.DataFrame:
#     """Load CSV or Parquet based on file suffix, with graceful fallback."""
#     suf = path.suffix.lower()
#     if suf in {".parquet", ".pq"}:
#         # Requires pyarrow or fastparquet
#         return pd.read_parquet(path)
#     if suf == ".csv":
#         return pd.read_csv(path)
#     # Fallback: try parquet then csv
#     try:
#         return pd.read_parquet(path)
#     except Exception:
#         return pd.read_csv(path)

# # ---------- Load data ----------
# imdb_cmu_df = smart_read(imdb_cmu_path)
# netflix_df  = smart_read(netflix_path)

# # ---------- Sanity checks ----------
# required_cols = {"year", "title_norm"}
# missing_a = required_cols - set(imdb_cmu_df.columns)
# missing_b = required_cols - set(netflix_df.columns)
# if missing_a:
#     raise ValueError(f"IMDb–CMU file missing columns: {sorted(missing_a)}")
# if missing_b:
#     raise ValueError(f"Netflix file missing columns: {sorted(missing_b)}")

# # Be tolerant of integer NA in MovieID
# if "MovieID" in netflix_df.columns:
#     netflix_df["MovieID"] = pd.to_numeric(netflix_df["MovieID"], errors="coerce").astype("Int64")

# # ---------- Merge ----------
# merged = imdb_cmu_df.merge(
#     netflix_df[["MovieID", "year", "title", "title_norm"]],
#     on=["title_norm", "year"],
#     how="left",
#     suffixes=("", "_netflix"),
# )

# # ---------- Audit ----------
# total = len(merged)
# matched = merged["MovieID"].notna().sum()
# unmatched = total - matched
# match_rate = (matched / total * 100.0) if total else 0.0

# with open(out_aud, "w", encoding="utf-8") as f:
#     f.write("IMDb+CMU ↔ Netflix Merge Audit\n")
#     f.write(f"Generated: {dt.datetime.now().isoformat(timespec='seconds')}\n\n")
#     f.write(f"IMDb+CMU source: {imdb_cmu_path}\n")
#     f.write(f"Netflix source : {netflix_path}\n\n")
#     f.write(f"Merged rows: {total}\n")
#     f.write(f"Matched Netflix titles: {matched}\n")
#     f.write(f"Unmatched IMDb+CMU titles: {unmatched}\n")
#     f.write(f"Match rate: {match_rate:.2f}%\n")
#     # Optional: quick column list
#     f.write(f"\nColumns: {list(merged.columns)}\n")

# # ---------- Save (CSV + Parquet) ----------
# merged.to_csv(out_csv, index=False, encoding="utf-8", lineterminator="\n")

# parquet_status = "saved"
# try:
#     merged.to_parquet(out_parq, index=False)  # needs pyarrow or fastparquet
# except Exception as e:
#     parquet_status = f"skipped — install 'pyarrow' or 'fastparquet' — {e}"

# print(" Merge complete.")
# print(f"   CSV saved   → {out_csv}")
# print(f"   Parquet     → {out_parq} ({parquet_status})")
# print(f"   Audit       → {out_aud}")


### 1.7 Keep only Netflix-overlapping titles

In [ ]:
# # === Filter to Netflix-matched + IMDb + CMU rows, and audit ===
# from pathlib import Path
# import pandas as pd
# import numpy as np
# import datetime as dt

# # --------- EDIT THESE PATHS ----------
# in_path = Path(r"D:\dataset_assignment2\dataset\main_dataset\merged_imdb_cmu_netflix.parquet")
# out_dir = Path(r"D:\dataset_assignment2\dataset\main_dataset")
# # -------------------------------------

# out_dir.mkdir(parents=True, exist_ok=True)
# out_csv  = out_dir / "imdb_cmu_netflix_only_v1.csv"
# out_parq = out_dir / "imdb_cmu_netflix_only_v1.parquet"
# out_aud  = out_dir / "imdb_cmu_netflix_only_v1.audit.txt"

# def smart_read(path: Path) -> pd.DataFrame:
#     s = path.suffix.lower()
#     if s in {".parquet", ".pq"}:
#         return pd.read_parquet(path)
#     if s == ".csv":
#         return pd.read_csv(path)
#     try:
#         return pd.read_parquet(path)
#     except Exception:
#         return pd.read_csv(path)

# # ---------- Load ----------
# df = smart_read(in_path)

# # ---------- Basic normalization for checks ----------
# def nonempty(series: pd.Series) -> pd.Series:
#     return series.astype(str).str.strip().replace({"": np.nan}).notna()

# before_rows = len(df)

# # Flags
# has_netflix = df["MovieID"].notna()
# has_imdb    = df["tconst"].notna()
# has_cmu     = df["plot_summary"].notna() & nonempty(df["plot_summary"])

# # Filter: Netflix + IMDb + CMU
# mask = has_netflix & has_imdb & has_cmu
# df_f = df.loc[mask].copy()
# after_rows = len(df_f)

# # ---------- Audit metrics ----------
# imdb_missing      = (~has_imdb).sum()
# cmu_missing_text  = (~has_cmu).sum()
# netflix_missing   = (~has_netflix).sum()

# # Optional: how many keep if you only required Netflix (for reference)
# kept_if_netflix_only = has_netflix.sum()

# # ---------- Save ----------
# df_f.to_csv(out_csv, index=False, encoding="utf-8", lineterminator="\n")
# parquet_status = "saved"
# try:
#     df_f.to_parquet(out_parq, index=False)
# except Exception as e:
#     parquet_status = f"skipped — install pyarrow/fastparquet — {e}"

# # ---------- Write audit ----------
# with open(out_aud, "w", encoding="utf-8") as f:
#     f.write("IMDb+CMU+Netflix OVERLAP — FILTER & AUDIT\n")
#     f.write(f"Generated: {dt.datetime.now().isoformat(timespec='seconds')}\n\n")
#     f.write(f"Source file: {in_path}\n\n")
#     f.write(f"Rows BEFORE: {before_rows}\n")
#     f.write(f" - Missing IMDb tconst:      {imdb_missing}\n")
#     f.write(f" - Missing CMU plot_summary: {cmu_missing_text}\n")
#     f.write(f" - Missing Netflix MovieID:  {netflix_missing}\n")
#     f.write(f"\nFilter logic: MovieID!=NA AND tconst!=NA AND plot_summary non-empty\n")
#     f.write(f"Rows kept (AFTER): {after_rows}\n")
#     f.write(f" - Kept if only Netflix match (for reference): {kept_if_netflix_only}\n")
#     f.write(f"\nColumns saved: {list(df_f.columns)}\n")

# print(" Saved filtered overlap dataset:")
# print("   CSV   →", out_csv)
# print("   PQ    →", out_parq, f"({parquet_status})")
# print("   AUDIT →", out_aud)


### 1.8 Fill missing genre and language fields from CMU metadata

In [ ]:
# # === Fill missing genre/language from CMU columns and audit ===
# from pathlib import Path
# import pandas as pd
# import numpy as np
# import datetime as dt

# # ---------- EDIT THESE ----------
# in_path  = Path(r"D:\dataset_assignment2\dataset\main_dataset\imdb_cmu_netflix_only_v1.parquet")
# out_dir  = Path(r"D:\dataset_assignment2\dataset\main_dataset\working_dataset")
# # --------------------------------

# out_dir.mkdir(parents=True, exist_ok=True)
# out_csv  = out_dir / "imdb_cmu_netflix_filled_v1.csv"
# out_parq = out_dir / "imdb_cmu_netflix_filled_v1.parquet"
# out_aud  = out_dir / "imdb_cmu_netflix_filled_v1.audit.txt"

# # --- Robust load ---
# def smart_read(path: Path) -> pd.DataFrame:
#     s = path.suffix.lower()
#     if s in {".parquet", ".pq"}:
#         return pd.read_parquet(path)
#     return pd.read_csv(path)

# df = smart_read(in_path)
# before_rows = len(df)

# # --- Helper for empty detection ---
# def empty_mask(s: pd.Series) -> pd.Series:
#     return s.astype(str).str.strip().replace(["", "nan", "None"], np.nan).isna()

# # --- Normalize empty values ---
# for col in ["genre", "cmu_genres", "language", "cmu_languages"]:
#     if col in df.columns:
#         df[col] = df[col].astype(str).str.strip().replace(["", "nan", "None"], np.nan)

# # --- Count empties before fill ---
# genre_missing_before = df["genre"].isna().sum()
# lang_missing_before  = df["language"].isna().sum()

# # --- Fill missing values ---
# df["genre"] = df["genre"].fillna(df["cmu_genres"])
# df["language"] = df["language"].fillna(df["cmu_languages"])

# # --- Normalize separators & casing ---
# def normalize_text(s):
#     if pd.isna(s): return np.nan
#     s = str(s).replace("|", ",").replace(";", ",")
#     s = ", ".join(sorted(set([x.strip().title() for x in s.split(",") if x.strip()])))
#     return s

# df["genre"] = df["genre"].apply(normalize_text)
# df["language"] = df["language"].apply(normalize_text)

# # --- Count empties after fill ---
# genre_missing_after = df["genre"].isna().sum()
# lang_missing_after  = df["language"].isna().sum()

# # --- Drop helper columns ---
# df = df.drop(columns=[c for c in ["cmu_genres", "cmu_languages"] if c in df.columns])

# # --- Save outputs ---
# df.to_csv(out_csv, index=False, encoding="utf-8", lineterminator="\n")
# parquet_status = "saved"
# try:
#     df.to_parquet(out_parq, index=False)
# except Exception as e:
#     parquet_status = f"skipped (need pyarrow/fastparquet) — {e}"

# # --- Write audit ---
# with open(out_aud, "w", encoding="utf-8") as f:
#     f.write("IMDb+CMU+Netflix — Fill Missing Genre/Language Audit\n")
#     f.write(f"Generated: {dt.datetime.now().isoformat(timespec='seconds')}\n\n")
#     f.write(f"Input file:  {in_path}\n")
#     f.write(f"Output dir:  {out_dir}\n\n")
#     f.write(f"Rows processed: {before_rows}\n")
#     f.write(f"Missing genre before fill:  {genre_missing_before}\n")
#     f.write(f"Missing genre after fill:   {genre_missing_after}\n")
#     f.write(f"Missing language before fill:  {lang_missing_before}\n")
#     f.write(f"Missing language after fill:   {lang_missing_after}\n\n")
#     f.write(f"Columns saved: {list(df.columns)}\n")

# print(" Genre/Language fill completed!")
# print(f"   CSV   → {out_csv}")
# print(f"   Parquet → {out_parq} ({parquet_status})")
# print(f"   Audit  → {out_aud}")


### 1.9 Extract Netflix ratings for overlapping movies

In [ ]:
# # === Netflix ratings extractor (filtered to overlap MovieIDs) ===
# from pathlib import Path
# import pandas as pd
# import numpy as np
# import datetime as dt

# # ---------------- EDIT THESE ----------------
# movies_path = Path(r"D:\dataset_assignment2\dataset\main_dataset\working_dataset\imdb_cmu_netflix_filled_v1.parquet")
# netflix_dir = Path(r"D:\dataset_assignment2\dataset\raw_data\netflix_data")
# out_dir     = Path(r"D:\dataset_assignment2\dataset\processed\ratings_netflix")
# # --------------------------------------------

# out_dir.mkdir(parents=True, exist_ok=True)
# out_parq = out_dir / "netflix_ratings_filtered.parquet"
# out_csv  = out_dir / "netflix_ratings_filtered.csv"
# out_avg  = out_dir / "netflix_ratings_avg.csv"
# out_aud  = out_dir / "netflix_ratings_extract.audit.txt"

# def smart_read_table(p: Path) -> pd.DataFrame:
#     suf = p.suffix.lower()
#     if suf in {".parquet", ".pq"}:
#         return pd.read_parquet(p)
#     return pd.read_csv(p)

# # 1) Load overlap MovieIDs from the movies file (no prior DF assumed)
# movies_df = smart_read_table(movies_path)
# if "MovieID" not in movies_df.columns:
#     raise ValueError("The movies file must contain a 'MovieID' column.")
# overlap_ids = set(pd.to_numeric(movies_df["MovieID"], errors="coerce").dropna().astype(int).unique())

# # 2) Stream-parse combined_data_1..4.txt keeping only overlap blocks
# rating_rows = []
# kept_movies = set()
# skipped_movies = 0
# kept_movies_count = 0
# kept_rows = 0

# def parse_one_file(txt_path: Path):
#     global kept_movies_count, kept_rows, skipped_movies
#     with txt_path.open("r", encoding="latin-1") as f:
#         current_mid = None
#         keep_block = False
#         for line in f:
#             line = line.strip()
#             if not line:
#                 continue
#             if line.endswith(":"):
#                 # New MovieID block header
#                 try:
#                     current_mid = int(line[:-1])
#                 except ValueError:
#                     current_mid = None
#                 keep_block = (current_mid in overlap_ids) if current_mid is not None else False
#                 if keep_block:
#                     kept_movies.add(current_mid)
#                     kept_movies_count += 1
#                 else:
#                     skipped_movies += 1
#                 continue
#             # rating line
#             if keep_block and current_mid is not None:
#                 # Format: UserID,Rating,Date
#                 parts = line.split(",")
#                 if len(parts) >= 3:
#                     uid, r, d = parts[0], parts[1], parts[2]
#                     rating_rows.append((current_mid, int(uid), float(r), d))
#                     kept_rows += 1

# # Parse all four files
# sources = [netflix_dir / f"combined_data_{i}.txt" for i in (1,2,3,4)]
# for p in sources:
#     if not p.exists():
#         print(f"Warning: Missing file: {p}")
#     else:
#         print(f"Parsing {p.name} ...")
#         parse_one_file(p)

# # 3) Build DataFrame
# ratings = pd.DataFrame(rating_rows, columns=["MovieID", "UserID", "Rating", "Date"])
# # keep Date for possible temporal split; also coerce to datetime safely
# ratings["Date"] = pd.to_datetime(ratings["Date"], errors="coerce")

# # 4) Save filtered ratings (Parquet + CSV)
# parquet_status = "saved"
# try:
#     ratings.to_parquet(out_parq, index=False)
# except Exception as e:
#     parquet_status = f"skipped (install pyarrow/fastparquet) — {e}"
# ratings.to_csv(out_csv, index=False, encoding="utf-8", lineterminator="\n")

# # 5) Aggregates per MovieID (for merging into movies)
# avg = (
#     ratings.groupby("MovieID", as_index=False)
#            .agg(avg_rating=("Rating","mean"),
#                 num_ratings=("Rating","count"),
#                 stdev_rating=("Rating","std"),
#                 first_date=("Date","min"),
#                 last_date=("Date","max"))
# )
# avg.to_csv(out_avg, index=False, encoding="utf-8", lineterminator="\n")

# # 6) Audit
# with open(out_aud, "w", encoding="utf-8") as f:
#     f.write("NETFLIX RATINGS — FILTERED EXTRACT (overlap MovieIDs)\n")
#     f.write(f"Generated: {dt.datetime.now().isoformat(timespec='seconds')}\n\n")
#     f.write(f"Movies source: {movies_path}\n")
#     f.write(f"Netflix dir :  {netflix_dir}\n")
#     f.write(f"Output dir  :  {out_dir}\n\n")
#     f.write(f"Overlap MovieIDs (unique): {len(overlap_ids)}\n")
#     f.write(f"Movies kept (blocks matched): {len(kept_movies)}\n")
#     f.write(f"Movie blocks skipped: {skipped_movies}\n")
#     f.write(f"Rating rows kept: {kept_rows}\n\n")
#     f.write(f"Saved ratings: {out_csv.name}, {out_parq.name} ({parquet_status})\n")
#     f.write(f"Saved per-movie averages: {out_avg.name}\n")
#     f.write(f"\nAverages columns: {list(avg.columns)}\n")

# print(" Done.")
# print(f"   Ratings CSV   → {out_csv}")
# print(f"   Ratings PQ    → {out_parq} ({parquet_status})")
# print(f"   Averages CSV  → {out_avg}")
# print(f"   Audit         → {out_aud}")


### 1.10 Add per-movie average ratings

In [ ]:
# # === Merge IMDb+CMU+Netflix metadata with per-movie average ratings ===
# from pathlib import Path
# import pandas as pd
# import datetime as dt

# # ---------- EDIT THESE ----------
# movies_path = Path(r"D:\dataset_assignment2\dataset\main_dataset\working_dataset\imdb_cmu_netflix_filled_v1.csv")
# ratings_avg_path = Path(r"D:\dataset_assignment2\dataset\processed\ratings_netflix\netflix_ratings_avg.csv")
# out_dir = Path(r"D:\dataset_assignment2\dataset\main_dataset\final_dataset")
# # --------------------------------

# out_dir.mkdir(parents=True, exist_ok=True)
# out_csv  = out_dir / "imdb_cmu_netflix_final_with_ratings.csv"
# out_parq = out_dir / "imdb_cmu_netflix_final_with_ratings.parquet"
# out_aud  = out_dir / "imdb_cmu_netflix_final_with_ratings.audit.txt"

# # --- Load movie dataset and ratings averages ---
# movies = pd.read_csv(movies_path)
# ratings_avg = pd.read_csv(ratings_avg_path)

# # --- Basic sanity checks ---
# assert "MovieID" in movies.columns, "MovieID missing in movies dataset"
# assert "MovieID" in ratings_avg.columns, "MovieID missing in ratings averages"

# # --- Merge on MovieID ---
# merged = movies.merge(
#     ratings_avg[["MovieID", "avg_rating", "num_ratings"]],
#     on="MovieID", how="left"
# )

# # --- Audit info ---
# total_movies = len(merged)
# matched_ratings = merged["avg_rating"].notna().sum()
# missing_ratings = total_movies - matched_ratings
# match_rate = matched_ratings / total_movies * 100

# # --- Save outputs ---
# merged.to_csv(out_csv, index=False, encoding="utf-8", lineterminator="\n")
# parquet_status = "saved"
# try:
#     merged.to_parquet(out_parq, index=False)
# except Exception as e:
#     parquet_status = f"skipped (install pyarrow/fastparquet) — {e}"

# # --- Write audit summary ---
# with open(out_aud, "w", encoding="utf-8") as f:
#     f.write("IMDb+CMU+Netflix — Final merge with average ratings\n")
#     f.write(f"Generated: {dt.datetime.now().isoformat(timespec='seconds')}\n\n")
#     f.write(f"Movies file:   {movies_path}\n")
#     f.write(f"Ratings file:  {ratings_avg_path}\n")
#     f.write(f"Output folder: {out_dir}\n\n")
#     f.write(f"Movies total:     {total_movies}\n")
#     f.write(f"Movies with rating: {matched_ratings}\n")
#     f.write(f"Movies missing rating: {missing_ratings}\n")
#     f.write(f"Match rate: {match_rate:.2f}%\n\n")
#     f.write(f"Columns saved: {list(merged.columns)}\n")

# print(" Merge complete.")
# print(f"   Saved CSV   → {out_csv}")
# print(f"   Saved PARQUET → {out_parq} ({parquet_status})")
# print(f"   Audit → {out_aud}")


### 1.11 Final dataset checks

In [ ]:

# import pandas as pd
# from pathlib import Path

# # --- EDIT THIS PATH ---
# in_path = Path(r"D:\dataset_assignment2\dataset\main_dataset\final_dataset\imdb_cmu_netflix_final_with_ratings.csv")

# # --- Load ---
# df = pd.read_csv(in_path)

# # --- Sanity check ---
# total = len(df)
# mismatch = df[df["title"] != df["title_netflix"]]
# mismatch_count = len(mismatch)
# pct = mismatch_count / total * 100

# print(" Title vs. Title_Netflix Sanity Check")
# print(f"Total movies: {total}")
# print(f"Mismatched titles: {mismatch_count} ({pct:.2f}%)")

# if mismatch_count > 0:
#     print("\nSample mismatches:")
#     display(mismatch.sample(min(5, mismatch_count))[["MovieID", "title", "title_netflix"]])
# else:
#     print("\n All titles match perfectly — you can safely drop 'title_netflix'.")


In [ ]:
# # === Final verification: title vs title_netflix match check ===
# import pandas as pd
# from pathlib import Path

# # --- EDIT THIS PATH if needed ---
# in_path = Path(r"D:\dataset_assignment2\dataset\main_dataset\final_dataset\imdb_cmu_netflix_final_with_ratings_titles_fixed.csv")

# # --- Load the dataset ---
# df = pd.read_csv(in_path)

# # --- Sanity check ---
# total = len(df)
# mismatch = df[df["title"] != df["title_netflix"]]
# mismatch_count = len(mismatch)
# pct = mismatch_count / total * 100

# print(" Final Title vs. Title_Netflix Match Check")
# print(f"Total movies: {total}")
# print(f"Mismatched titles: {mismatch_count} ({pct:.2f}%)")

# if mismatch_count > 0:
#     print("\nWarning: Sample mismatches:")
#     display(mismatch.sample(min(5, mismatch_count))[["MovieID", "title", "title_netflix"]])
# else:
#     print("\n All titles now perfectly match Netflix titles — you can safely drop 'title_netflix'.")


In [ ]:
# import pandas as pd
# from pathlib import Path

# # --- Path to the current file ---
# in_path = Path(r"D:\dataset_assignment2\dataset\main_dataset\final_dataset\imdb_cmu_netflix_final_with_ratings_titles_fixed.csv")

# # --- Load the dataset ---
# df = pd.read_csv(in_path)

# # --- Drop the redundant column ---
# df.drop(columns=["title_netflix"], inplace=True, errors="ignore")

# # --- Save cleaned dataset ---
# out_path = in_path.with_name("imdb_cmu_netflix_final_readyM.csv")
# df.to_csv(out_path, index=False, encoding="utf-8", lineterminator="\n")

# print(f" 'title_netflix' column dropped successfully.")
# print(f"💾 Clean dataset saved as:\n{out_path}")


In [ ]:
# path = Path(r"D:\dataset_assignment2\dataset\main_dataset\final_dataset\imdb_cmu_netflix_final_readyM.csv")

# dataset = pd.read_csv(path)

# print(dataset.columns)


In [ ]:
# # === Pre-check and cleaning before Feature Engineering ===
# import pandas as pd
# import numpy as np
# from pathlib import Path

# # --- EDIT THIS PATH ---
# in_path = Path(r"D:\dataset_assignment2\dataset\main_dataset\final_dataset\imdb_cmu_netflix_final_readyM.csv")

# # --- Load dataset ---
# df = pd.read_csv(in_path)

# print(" Dataset loaded successfully.")
# print(f"Shape: {df.shape}")
# print("\nColumns:", df.columns.tolist())

# # --- Basic info ---
# print("\n=== Dataset Info ===")
# display(df.info())

# # --- Quick NaN/Empty check for all columns ---
# print("\n=== Missing / Empty Values ===")
# missing_report = df.isna().sum().to_frame("NaN_count")
# missing_report["Empty_string_count"] = (df.applymap(lambda x: isinstance(x, str) and x.strip() == "")).sum()
# missing_report["Total_missing"] = missing_report["NaN_count"] + missing_report["Empty_string_count"]
# display(missing_report)

# # --- Datatype check ---
# print("\n=== Datatype Summary ===")
# display(df.dtypes.to_frame("dtype"))

# # --- Check specific feature columns for potential issues ---
# feature_cols = ["genre", "language", "directors", "writers", "stars", "plot_summary"]
# for col in feature_cols:
#     unique_vals = df[col].nunique(dropna=True)
#     nan_count = df[col].isna().sum()
#     empty_count = (df[col].astype(str).str.strip() == "").sum()
#     print(f"\n {col}:")
#     print(f" - Unique non-null entries: {unique_vals}")
#     print(f" - NaN: {nan_count} | Empty: {empty_count}")

# # --- Normalize missing and empty entries to safe strings ---
# for col in feature_cols:
#     df[col] = df[col].fillna("").astype(str).str.strip()

# # --- Sanity: are there any rows fully blank in all text features? ---
# blank_rows = (df[feature_cols].apply(lambda x: x.str.len() == 0)).all(axis=1).sum()
# print(f"\nWarning: Movies with ALL empty feature text fields: {blank_rows} (these might be dropped later)")

# # --- Clean up potential duplicates ---
# before = len(df)
# df = df.drop_duplicates(subset=["MovieID"], keep="first")
# after = len(df)
# if before != after:
#     print(f"\nWarning: Removed {before - after} duplicate MovieIDs.")

# # --- Confirm numeric types for ratings ---
# df["avg_rating"] = pd.to_numeric(df["avg_rating"], errors="coerce")
# df["num_ratings"] = pd.to_numeric(df["num_ratings"], errors="coerce")

# print("\n Data types normalized and text columns cleaned.")
# print(f"Final shape after cleaning: {df.shape}")

# # --- Save the cleaned preprocessed version ---
# out_path = in_path.with_name("imdb_cmu_netflix_prechecked.csv")
# df.to_csv(out_path, index=False, encoding="utf-8", lineterminator="\n")

# print(f"\n💾 Clean pre-checked dataset saved as:\n{out_path}")


In [ ]:
# # === Fill final small NaN gaps before encoding ===
# fill_cols = ["language", "directors", "writers", "stars"]
# for col in fill_cols:
#     df[col] = df[col].fillna("Unknown").astype(str).str.strip()

# # Ensure no residual NaNs in key text features
# df["plot_summary"] = df["plot_summary"].fillna("").astype(str)
# df["genre"] = df["genre"].fillna("").astype(str)

# # Save final feature-engineering-ready dataset
# out_path = in_path.with_name("imdb_cmu_netflix_feature_ready.csv")
# df.to_csv(out_path, index=False, encoding="utf-8", lineterminator="\n")

# print(" All feature columns filled and normalized.")
# print(f"💾 Saved feature-ready dataset to:\n{out_path}")


In [ ]:
# # === Deep diagnostic check before feature engineering ===
# import pandas as pd
# import numpy as np
# from pathlib import Path

# # --- Edit path ---
# in_path = Path(r"D:\dataset_assignment2\dataset\main_dataset\final_dataset\imdb_cmu_netflix_feature_ready.csv")

# df = pd.read_csv(in_path)

# print(" Loaded:", in_path.name)
# print(f"Shape: {df.shape}\n")

# # -------------------------------------------------------------------------
# # 1️⃣ Basic structure info
# print("=== DataFrame Info ===")
# display(df.info())

# # -------------------------------------------------------------------------
# # 2️⃣ Missing values overview
# print("\n=== Missing value summary ===")
# na_summary = df.isna().sum().to_frame("NaN_count")
# na_summary["Empty_string_count"] = (df.applymap(lambda x: isinstance(x, str) and x.strip() == "")).sum()
# na_summary["Total_missing"] = na_summary["NaN_count"] + na_summary["Empty_string_count"]
# display(na_summary)

# # -------------------------------------------------------------------------
# # 3️⃣ Check column datatypes
# print("\n=== Datatypes ===")
# display(df.dtypes.to_frame("dtype"))

# # -------------------------------------------------------------------------
# # 4️⃣ Validate numeric columns
# num_cols = ["avg_rating", "num_ratings", "year"]
# for col in num_cols:
#     nonnum = df[pd.to_numeric(df[col], errors="coerce").isna()].shape[0]
#     print(f" {col}: {nonnum} non-numeric rows")
# df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")

# # -------------------------------------------------------------------------
# # 5️⃣ Verify string/text columns integrity
# text_cols = ["genre","language","directors","writers","stars","plot_summary"]
# for col in text_cols:
#     print(f"\n Checking {col}:")
#     nan = df[col].isna().sum()
#     empt = (df[col].astype(str).str.strip() == "").sum()
#     print(f" - NaN: {nan} | Empty: {empt}")
#     print(f" - Example values: {df[col].dropna().astype(str).sample(3, random_state=1).tolist()}")

# # -------------------------------------------------------------------------
# # 6️⃣ Detect potential outliers in ratings
# print("\n=== Ratings summary ===")
# print(df[["avg_rating", "num_ratings"]].describe())

# # -------------------------------------------------------------------------
# # 7️⃣ Check duplicates
# dups = df.duplicated(subset=["MovieID"]).sum()
# print(f"\n=== Duplicates check ===")
# print(f"Duplicate MovieIDs: {dups}")

# # -------------------------------------------------------------------------
# # 8️⃣ Ensure no all-empty feature rows
# feature_cols = ["genre","language","directors","writers","stars","plot_summary"]
# blank_rows = (df[feature_cols].apply(lambda x: x.str.len() == 0)).all(axis=1).sum()
# print(f"\nWarning: Movies with ALL empty feature fields: {blank_rows}")

# # -------------------------------------------------------------------------
# # 9️⃣ Final sanity note
# if blank_rows == 0 and dups == 0:
#     print("\n Dataset is clean and fully safe for one-hot and TF-IDF.")
# else:
#     print("\nWarning: Review warnings above before proceeding.")

# # Optional save: normalized text fields stripped
# for col in text_cols:
#     df[col] = df[col].astype(str).str.strip()
# out_path = in_path.with_name("imdb_cmu_netflix_verified.csv")
# df.to_csv(out_path, index=False, encoding="utf-8", lineterminator="\n")
# print(f"\n💾 Verified dataset saved to: {out_path}")


## Part 2: Feature engineering

### Feature engineering overview

In [ ]:
import pandas as pd

input_path = FINAL_DATASET_PATH
imdb_cmu_net_rat_dataset = pd.read_csv(input_path)

print(imdb_cmu_net_rat_dataset.columns)
print(imdb_cmu_net_rat_dataset.shape)


In [ ]:

movies_df = imdb_cmu_net_rat_dataset.copy()

print("Working copy created:")
print("Shape:", movies_df.shape)
print("Columns:", movies_df.columns.tolist())


### 2.1 Writer one-hot encoding

### 2.2 Director one-hot encoding

### 2.3 Genre one-hot encoding

### 2.4 Language one-hot encoding

### 2.5 Star/cast vectorization

### 2.6 Manual metadata feature matrix

### 2.7 Plot summary TF-IDF features

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter
import re

class ManualFeatureVectorizer:
    """
    Manual implementation of feature vectorization for movie recommendation system.
    This class handles one-hot encoding and custom vectorization without using sklearn.
    """
    
    def __init__(self):
        self.writer_vocab = {}
        self.director_vocab = {}
        self.genre_vocab = {}
        self.language_vocab = {}
        self.star_vocab = {}
        
    def fit(self, df):
        """
        Build vocabulary dictionaries from the dataset.
        Each unique value gets assigned an index.
        
        Args:
            df: DataFrame with columns: writers, directors, stars, genre, language
        """
        # Build writer vocabulary
        all_writers = set()
        for writers in df['writers'].fillna(''):
            if writers:
                writer_list = [w.strip() for w in str(writers).split(',')]
                all_writers.update(writer_list)
        self.writer_vocab = {writer: idx for idx, writer in enumerate(sorted(all_writers))}
        
        # Build director vocabulary
        all_directors = set()
        for directors in df['directors'].fillna(''):
            if directors:
                director_list = [d.strip() for d in str(directors).split(',')]
                all_directors.update(director_list)
        self.director_vocab = {director: idx for idx, director in enumerate(sorted(all_directors))}
        
        # Build genre vocabulary
        all_genres = set()
        for genres in df['genre'].fillna(''):
            if genres:
                genre_list = [g.strip() for g in str(genres).split(',')]
                all_genres.update(genre_list)
        self.genre_vocab = {genre: idx for idx, genre in enumerate(sorted(all_genres))}
        
        # Build language vocabulary
        all_languages = set()
        for languages in df['language'].fillna(''):
            if languages:
                lang_list = [l.strip() for l in str(languages).split(',')]
                all_languages.update(lang_list)
        self.language_vocab = {lang: idx for idx, lang in enumerate(sorted(all_languages))}
        
        # Build star vocabulary (top 500 most common words from all star names)
        all_star_text = ' '.join(df['stars'].fillna('').astype(str))
        # Tokenize: split by comma and spaces, clean
        words = re.findall(r'\b[a-zA-Z]+\b', all_star_text.lower())
        # Get top 500 most common words
        word_counts = Counter(words)
        top_words = [word for word, count in word_counts.most_common(500)]
        self.star_vocab = {word: idx for idx, word in enumerate(top_words)}
        
        print(f"Vocabularies built:")
        print(f"  Writers: {len(self.writer_vocab)}")
        print(f"  Directors: {len(self.director_vocab)}")
        print(f"  Genres: {len(self.genre_vocab)}")
        print(f"  Languages: {len(self.language_vocab)}")
        print(f"  Star words: {len(self.star_vocab)}")
        
    def transform_writers(self, writers_str):
        """
        Convert writers to one-hot encoding vector.
        
        Logic:
        1. Split the writers string by comma
        2. Create a zero vector of length = number of unique writers
        3. Set index to 1 for each writer present in the movie
        
        Args:
            writers_str: String of comma-separated writer names
            
        Returns:
            numpy array of shape (vocab_size,) with 1s at writer positions
        """
        vector = np.zeros(len(self.writer_vocab))
        
        if pd.isna(writers_str) or not writers_str:
            return vector
            
        writer_list = [w.strip() for w in str(writers_str).split(',')]
        for writer in writer_list:
            if writer in self.writer_vocab:
                idx = self.writer_vocab[writer]
                vector[idx] = 1
                
        return vector
    
    def transform_directors(self, directors_str):
        """
        Convert directors to one-hot encoding vector.
        
        Logic: Same as writers - binary encoding where 1 indicates presence
        
        Args:
            directors_str: String of comma-separated director names
            
        Returns:
            numpy array of shape (vocab_size,) with 1s at director positions
        """
        vector = np.zeros(len(self.director_vocab))
        
        if pd.isna(directors_str) or not directors_str:
            return vector
            
        director_list = [d.strip() for d in str(directors_str).split(',')]
        for director in director_list:
            if director in self.director_vocab:
                idx = self.director_vocab[director]
                vector[idx] = 1
                
        return vector
    
    def transform_genres(self, genre_str):
        """
        Convert genres to one-hot encoding vector.
        
        Logic: Multi-label one-hot encoding (multiple genres can be 1)
        
        Args:
            genre_str: String of comma-separated genres
            
        Returns:
            numpy array of shape (vocab_size,) with 1s at genre positions
        """
        vector = np.zeros(len(self.genre_vocab))
        
        if pd.isna(genre_str) or not genre_str:
            return vector
            
        genre_list = [g.strip() for g in str(genre_str).split(',')]
        for genre in genre_list:
            if genre in self.genre_vocab:
                idx = self.genre_vocab[genre]
                vector[idx] = 1
                
        return vector
    
    def transform_languages(self, language_str):
        """
        Convert languages to one-hot encoding vector.
        
        Logic: Multi-label one-hot encoding
        
        Args:
            language_str: String of comma-separated languages
            
        Returns:
            numpy array of shape (vocab_size,) with 1s at language positions
        """
        vector = np.zeros(len(self.language_vocab))
        
        if pd.isna(language_str) or not language_str:
            return vector
            
        lang_list = [l.strip() for l in str(language_str).split(',')]
        for lang in lang_list:
            if lang in self.language_vocab:
                idx = self.language_vocab[lang]
                vector[idx] = 1
                
        return vector
    
    def transform_stars(self, stars_str):
        """
        Convert star names to 500-length vector using bag-of-words approach.
        
        Logic:
        1. Concatenate all star names into one sentence
        2. Tokenize into words (split by space/comma, lowercase)
        3. Create a 500-length vector based on top 500 most common words
        4. Count frequency of each word in the star list
        5. Normalize by the total number of words to get term frequency
        
        This is essentially a manual TF (Term Frequency) vectorization.
        
        Args:
            stars_str: String of comma-separated star names
            
        Returns:
            numpy array of shape (500,) with normalized word frequencies
        """
        vector = np.zeros(500)
        
        if pd.isna(stars_str) or not stars_str:
            return vector
            
        # Tokenize: extract words, convert to lowercase
        words = re.findall(r'\b[a-zA-Z]+\b', str(stars_str).lower())
        
        if not words:
            return vector
        
        # Count word frequencies
        word_counts = Counter(words)
        total_words = len(words)
        
        # Fill vector with normalized frequencies
        for word, count in word_counts.items():
            if word in self.star_vocab:
                idx = self.star_vocab[word]
                # Normalize by total words (term frequency)
                vector[idx] = count / total_words
                
        return vector
    
    def transform_movie(self, row):
        """
        Transform a single movie row into complete feature vector.
        
        Args:
            row: pandas Series with movie features
            
        Returns:
            Concatenated numpy array of all features
        """
        writer_vec = self.transform_writers(row['writers'])
        director_vec = self.transform_directors(row['directors'])
        genre_vec = self.transform_genres(row['genre'])
        language_vec = self.transform_languages(row['language'])
        star_vec = self.transform_stars(row['stars'])
        
        # Concatenate all feature vectors
        full_vector = np.concatenate([
            writer_vec,
            director_vec,
            genre_vec,
            language_vec,
            star_vec
        ])
        
        return full_vector
    
    def transform(self, df):
        """
        Transform entire dataframe into feature matrix.
        
        Args:
            df: DataFrame with movie features
            
        Returns:
            numpy array of shape (n_movies, total_features)
        """
        feature_matrix = []
        
        for idx, row in df.iterrows():
            feature_vector = self.transform_movie(row)
            feature_matrix.append(feature_vector)
            
            # Progress indicator
            if (idx + 1) % 100 == 0:
                print(f"Processed {idx + 1} movies...")
        
        return np.array(feature_matrix)


In [ ]:
# Initialize the manual feature vectorizer
manual_vectorizer = ManualFeatureVectorizer()

# Build vocabularies from the dataset
manual_vectorizer.fit(movies_df)  # movies_df is the working DataFrame

# Transform metadata into the manual feature matrix
manual_feature_matrix = manual_vectorizer.transform(movies_df)

# Check the result
print("Manual feature matrix shape:", manual_feature_matrix.shape)


In [ ]:
# Save the manual feature matrix for later use
np.save('manual_feature_matrix.npy', manual_feature_matrix)


In [ ]:
manual_feature_matrix = np.load('manual_feature_matrix.npy')
print("manual feature matrix shape:", manual_feature_matrix.shape)


### 2.8 Reload generated feature matrices

In [ ]:
import numpy as np
from collections import Counter
import re
from sklearn.feature_extraction.text import TfidfVectorizer

# Step 1: Define a basic stopword set for cleaning
STOPWORDS = set([
    'the', 'of', 'and', 'a', 'in', 'to', 'is', 'with', 'on', 'for', 'by', 'an', 
    'as', 'at', 'from', 'that', 'this', 'it', 'its', 'are', 'was', 'be', 'he', 
    'she', 'his', 'her', 'or', 'but', 'not', 'which', 'who', 'they', 'their', 
    'have', 'has', 'we', 'you', 'i', 'my', 'all', 'can', 'will', 'would', 'been', 
    'more', 'one', 'about', 'after', 'so', 'when', 'there', 'into', 'also', 'than',
    'out', 'up', 'if'
])

def clean_text(text):
    """Lowercase, remove punctuation, remove stopwords."""
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    tokens = text.split()
    tokens = [w for w in tokens if w not in STOPWORDS]
    cleaned = " ".join(tokens)
    return cleaned

# Step 2: Make a list of cleaned plot summaries
plot_texts = movies_df["plot_summary"].fillna('').astype(str).tolist()
cleaned_texts = [clean_text(t) for t in plot_texts]

# Step 3: Build Zipf's law-based vocabulary (top 1000 most frequent words)
all_words = []
for t in cleaned_texts:
    all_words.extend(t.split())
common_word_counts = Counter(all_words)
zipf_vocab = [word for word, count in common_word_counts.most_common(1000)]

# Step 4: Compute TF-IDF matrix (length 1000 per movie)
vectorizer = TfidfVectorizer(vocabulary=zipf_vocab)
tfidf_matrix = vectorizer.fit_transform(cleaned_texts).toarray()

# Step 5: Save matrix for later use/concatenation
np.save('tfidf_matrix.npy', tfidf_matrix)


In [ ]:
tf_idf_matrix = np.load('tfidf_matrix.npy')
print("TF-IDF matrix shape:", tfidf_matrix.shape)


### 2.9 Build the full feature matrix

In [ ]:
import numpy as np

# The number of columns will be (manual feature size) + 1000 (for TF-IDF),

# Concatenate features along columns
full_feature_matrix = np.concatenate([manual_feature_matrix, tfidf_matrix], axis=1)

# Print to check
print("Full feature matrix shape:", full_feature_matrix.shape)

# Save for later reuse
np.save('full_feature_matrix.npy', full_feature_matrix)


## Part 3: Similarity computation

### 3.1 Movie-to-movie cosine similarity

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Load from disk if needed
# full_feature_matrix = np.load('full_feature_matrix.npy')

# Compute cosine similarity (movie-to-movie)
cosine_sim_matrix = cosine_similarity(full_feature_matrix)  # shape: (n_movies, n_movies)

# Save for later reuse
np.save('cosine_sim_matrix.npy', cosine_sim_matrix)
print("Cosine similarity matrix computed. Shape:", cosine_sim_matrix.shape)


### 3.2 Top-10 similar movie function

In [ ]:
def get_top_similar_movies(movie_title, movies_df, cosine_sim_matrix, top_n=10):
    """
    Given a movie title, returns the top N most similar movies (excluding itself).
    """
    # Find index of the given movie title
    indices = movies_df.index[movies_df['title'] == movie_title].tolist()
    if not indices:
        print(f"Movie '{movie_title}' not found.")
        return []
    idx = indices[0]
    # Get similarity scores
    sim_scores = cosine_sim_matrix[idx]
    # Get top N indices (excluding the movie itself)
    top_indices = sim_scores.argsort()[::-1][1:top_n+1]
    results = movies_df.iloc[top_indices][['title', 'year']].copy()
    results['similarity'] = sim_scores[top_indices]
    print(f"Top {top_n} movies similar to '{movie_title}':")
    print(results)
    return results

# Example usage:
# get_top_similar_movies("The Shawshank Redemption", movies_df, cosine_sim_matrix, top_n=10)


In [ ]:
# Ensure 'year' column in movies_df is numeric
movies_df["year"] = pd.to_numeric(movies_df["year"], errors="coerce")

# Drop invalid or missing years
valid_years = movies_df["year"].dropna().astype(int)

# Compute min and max years
min_year = valid_years.min()
max_year = valid_years.max()

print(f" Movie Release Year Range in Dataset:")
print(f"From {min_year} to {max_year}")
print(f"Total movies with valid year info: {len(valid_years)}")


### 3.3 Example query

In [ ]:

movie_title_input = input("Enter movie title: ")

# Call the recommendation function with user input
get_top_similar_movies(movie_title_input, movies_df, cosine_sim_matrix, top_n=10)


## Part 4: Content-based evaluation and visualization

### 4.1 Load Netflix training and probe/test ratings

In [ ]:
r_path = NETFLIX_RATINGS_PATH
netflix_ratings_filtered = pd.read_csv(r_path)
print(netflix_ratings_filtered.columns)
print(netflix_ratings_filtered.shape)


In [ ]:
p_path = PROBE_RATINGS_PATH
probe_filtered = pd.read_csv(p_path)
print(probe_filtered.columns)
print(probe_filtered.shape)


In [ ]:
print(movies_df.columns)


### 4.2 Generate sample content-based recommendations

• Show sample recommendations for at least 5 input movies for a given user x.

### 4.3 Content-based recommendation examples

In [ ]:
# Pick 5 sample movie titles from the movie metadata
sample_titles = ["Shawshank", "Godfather", "Matrix", "Toy Story", "Titanic"]

def get_content_recommendations(movie_title, movies_df, cosine_sim_matrix, top_n=10):
    # Find the movie by title (partial match)
    matches = movies_df[movies_df['title'].str.contains(movie_title, case=False, na=False)]
    if matches.empty:
        print(f"Movie titled '{movie_title}' not found.")
        return
    idx = matches.index[0]
    sim_scores = cosine_sim_matrix[idx]
    sim_indices = np.argsort(sim_scores)[::-1][1:top_n+1]  # Skip self
    print(f"\nRecommendations for '{movies_df.loc[idx, 'title']}' ({movies_df.loc[idx, 'year']}):")
    for rank, rec_idx in enumerate(sim_indices, 1):
        rec_title = movies_df.loc[rec_idx, 'title']
        rec_year = movies_df.loc[rec_idx, 'year']
        rec_score = sim_scores[rec_idx]
        print(f"{rank:2d}. {rec_title} ({rec_year}) - Similarity: {rec_score:.3f}")

for title in sample_titles:
    get_content_recommendations(title, movies_df, cosine_sim_matrix, top_n=10)


### 4.4 Content-based RMSE evaluation

In [ ]:
from sklearn.metrics import mean_squared_error
import numpy as np

def predict_content_rating(user_id, movie_id, movies_df, cosine_sim_matrix, train_ratings, global_avg, top_k=30):
    # Map MovieID to index in movies_df
    movie_id_map = {mid: idx for idx, mid in enumerate(movies_df['MovieID'])}
    if movie_id not in movie_id_map:
        return global_avg
    tgt_idx = movie_id_map[movie_id]
    # Get all movies user has rated
    user_ratings = train_ratings[train_ratings['UserID'] == user_id]
    similarities = []
    ratings = []
    for _, row in user_ratings.iterrows():
        rated_id = row['MovieID']
        if rated_id in movie_id_map:
            rated_idx = movie_id_map[rated_id]
            sim = cosine_sim_matrix[tgt_idx, rated_idx]
            if sim > 0:
                similarities.append(sim)
                ratings.append(row['Rating'])
    if not similarities:
        return global_avg
    similarities = np.array(similarities)
    ratings = np.array(ratings)
    # Optionally keep only top_k
    if len(similarities) > top_k:
        top_idx = np.argsort(similarities)[-top_k:]
        similarities = similarities[top_idx]
        ratings = ratings[top_idx]
    return np.sum(similarities * ratings) / np.sum(similarities)

# Run RMSE test (sample for speed)
global_avg = netflix_ratings_filtered['Rating'].mean()
test_subset = probe_filtered.sample(n=20000, random_state=42)

y_true = []
y_pred = []

for ix, row in test_subset.iterrows():
    pred = predict_content_rating(
        row['UserID'], row['MovieID'], movies_df, cosine_sim_matrix,
        netflix_ratings_filtered, global_avg, top_k=30
    )
    y_true.append(row['Rating'])
    y_pred.append(pred)

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
print(f"Test RMSE (content-based): {rmse:.4f}")


### 4.5 Actual vs predicted rating visualization

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 6))
plt.scatter(y_true, y_pred, alpha=0.2, s=10, color='blue')
plt.plot([1, 5], [1, 5], linestyle='--', color='red', label='Perfect Prediction')
plt.xlabel("Actual Rating")
plt.ylabel("Predicted Rating")
plt.title("Actual vs Predicted Ratings")
plt.legend()
plt.show()


## Assignment 2B: Collaborative filtering

### 5.1 Load Netflix movie-rating data

In [ ]:
print(probe_filtered.columns)
print(probe_filtered.shape)


In [ ]:
print(netflix_ratings_filtered.columns)
print(netflix_ratings_filtered.shape)


### 5.2 User-user collaborative filtering

In [ ]:
"""
Assignment 2B.1: User-User Collaborative Filtering
Uses user similarity to predict ratings
"""

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

class UserUserCollaborativeFiltering:
    """
    User-User Collaborative Filtering Recommender System.
    
    Logic: "Users who are similar to you liked these movies"
    - Find users with similar rating patterns
    - Predict based on what similar users rated
    """
    
    def __init__(self, top_k_neighbors=50):
        """
        Args:
            top_k_neighbors: Number of most similar users to consider
        """
        self.top_k = top_k_neighbors
        self.rating_matrix = None
        self.user_id_map = {}
        self.movie_id_map = {}
        self.reverse_user_map = {}
        self.reverse_movie_map = {}
        self.user_means = None
        self.global_mean = None
        
    def create_rating_matrix(self, ratings_df):
        """
        Create user-item rating matrix from ratings dataframe.
        
        Args:
            ratings_df: DataFrame with columns [UserID, MovieID, Rating]
        
        Returns:
            Sparse matrix of shape (n_users, n_movies)
        """
        print("Creating rating matrix...")
        
        # Create ID mappings
        unique_users = ratings_df['UserID'].unique()
        unique_movies = ratings_df['MovieID'].unique()
        
        self.user_id_map = {uid: idx for idx, uid in enumerate(unique_users)}
        self.movie_id_map = {mid: idx for idx, mid in enumerate(unique_movies)}
        
        # Reverse mappings
        self.reverse_user_map = {idx: uid for uid, idx in self.user_id_map.items()}
        self.reverse_movie_map = {idx: mid for mid, idx in self.movie_id_map.items()}
        
        # Create sparse matrix indices
        row_indices = [self.user_id_map[uid] for uid in ratings_df['UserID']]
        col_indices = [self.movie_id_map[mid] for mid in ratings_df['MovieID']]
        ratings_data = ratings_df['Rating'].values
        
        # Create sparse matrix
        self.rating_matrix = csr_matrix(
            (ratings_data, (row_indices, col_indices)),
            shape=(len(unique_users), len(unique_movies))
        )
        
        print(f" Rating matrix created: {self.rating_matrix.shape}")
        print(f"   Users: {len(unique_users):,}")
        print(f"   Movies: {len(unique_movies):,}")
        print(f"   Ratings: {len(ratings_data):,}")
        print(f"   Sparsity: {100 * (1 - len(ratings_data) / (len(unique_users) * len(unique_movies))):.2f}%")
        
        return self.rating_matrix
    
    def fit(self, ratings_df):
        """
        Fit the model by creating rating matrix and computing user statistics.
        
        Args:
            ratings_df: Training ratings DataFrame [UserID, MovieID, Rating]
        """
        print("\n" + "="*80)
        print("FITTING USER-USER COLLABORATIVE FILTERING MODEL")
        print("="*80)
        
        # Create rating matrix
        self.create_rating_matrix(ratings_df)
        
        # Calculate user mean ratings (for mean-centering)
        print("\nCalculating user statistics...")
        self.user_means = np.array(self.rating_matrix.mean(axis=1)).flatten()
        self.global_mean = ratings_df['Rating'].mean()
        
        print(f" Global mean rating: {self.global_mean:.4f}")
        print(f" Model fitted successfully!")
        print("="*80 + "\n")
    
    def compute_user_similarity(self, user_idx):
        """
        Compute similarity between a user and all other users.
        
        Uses Pearson correlation (better than cosine for ratings):
        - Accounts for different rating scales (some users rate high, others low)
        - Mean-centered ratings
        
        Args:
            user_idx: Index of the user in rating matrix
            
        Returns:
            Array of similarities with all users
        """
        # Get user's ratings (sparse row)
        user_ratings = self.rating_matrix[user_idx].toarray().flatten()
        
        # Mean-center the ratings (Pearson correlation)
        user_mean = self.user_means[user_idx]
        user_ratings_centered = user_ratings - user_mean
        user_ratings_centered[user_ratings == 0] = 0  # Keep zeros as zeros
        
        # Compute similarity with all users
        similarities = []
        
        for other_idx in range(self.rating_matrix.shape[0]):
            if other_idx == user_idx:
                similarities.append(0)  # Don't compare with self
                continue
            
            other_ratings = self.rating_matrix[other_idx].toarray().flatten()
            other_mean = self.user_means[other_idx]
            other_ratings_centered = other_ratings - other_mean
            other_ratings_centered[other_ratings == 0] = 0
            
            # Find common rated movies (both non-zero)
            common_mask = (user_ratings > 0) & (other_ratings > 0)
            
            if np.sum(common_mask) == 0:
                similarities.append(0)
                continue
            
            # Pearson correlation on common movies
            user_common = user_ratings_centered[common_mask]
            other_common = other_ratings_centered[common_mask]
            
            # Compute correlation
            num = np.sum(user_common * other_common)
            denom = np.sqrt(np.sum(user_common**2)) * np.sqrt(np.sum(other_common**2))
            
            if denom == 0:
                similarities.append(0)
            else:
                similarities.append(num / denom)
        
        return np.array(similarities)
    
    def predict_rating(self, user_id, movie_id):
        """
        Predict rating for a user-movie pair using User-User CF.
        
        Algorithm:
        1. Find K most similar users who rated this movie
        2. Use weighted average of their ratings
        3. Formula: r̂(u,i) = r̄_u + Σ[sim(u,v) × (r(v,i) - r̄_v)] / Σ|sim(u,v)|
        
        Args:
            user_id: User ID
            movie_id: Movie ID
            
        Returns:
            Predicted rating (1-5)
        """
        # Check if user/movie exist
        if user_id not in self.user_id_map:
            return self.global_mean
        if movie_id not in self.movie_id_map:
            return self.global_mean
        
        user_idx = self.user_id_map[user_id]
        movie_idx = self.movie_id_map[movie_id]
        
        # Get user's mean rating
        user_mean = self.user_means[user_idx]
        
        # Compute similarities with all users (expensive!)
        # In practice, use precomputed similarities or approximate methods
        user_similarities = self.compute_user_similarity(user_idx)
        
        # Find users who rated this movie
        movie_ratings = self.rating_matrix[:, movie_idx].toarray().flatten()
        rated_mask = movie_ratings > 0
        
        if np.sum(rated_mask) == 0:
            return user_mean  # No one rated this movie
        
        # Filter to users who rated this movie
        similar_users_similarities = user_similarities[rated_mask]
        similar_users_ratings = movie_ratings[rated_mask]
        similar_users_means = self.user_means[rated_mask]
        
        # Get top K most similar users
        if len(similar_users_similarities) > self.top_k:
            top_k_indices = np.argsort(similar_users_similarities)[-self.top_k:]
            similar_users_similarities = similar_users_similarities[top_k_indices]
            similar_users_ratings = similar_users_ratings[top_k_indices]
            similar_users_means = similar_users_means[top_k_indices]
        
        # Remove negative similarities (dissimilar users)
        positive_mask = similar_users_similarities > 0
        
        if np.sum(positive_mask) == 0:
            return user_mean
        
        similar_users_similarities = similar_users_similarities[positive_mask]
        similar_users_ratings = similar_users_ratings[positive_mask]
        similar_users_means = similar_users_means[positive_mask]
        
        # Weighted average prediction (mean-centered)
        numerator = np.sum(similar_users_similarities * (similar_users_ratings - similar_users_means))
        denominator = np.sum(np.abs(similar_users_similarities))
        
        if denominator == 0:
            return user_mean
        
        predicted_rating = user_mean + (numerator / denominator)
        
        # Clip to valid range [1, 5]
        predicted_rating = np.clip(predicted_rating, 1, 5)
        
        return predicted_rating
    
    def recommend_for_user(self, user_id, n_recommendations=10, exclude_rated=True):
        """
        Generate top N movie recommendations for a user.
        
        Args:
            user_id: User ID
            n_recommendations: Number of recommendations to return
            exclude_rated: If True, don't recommend already rated movies
            
        Returns:
            List of (movie_id, predicted_rating) tuples
        """
        if user_id not in self.user_id_map:
            return []
        
        user_idx = self.user_id_map[user_id]
        
        # Get movies user has already rated
        user_ratings = self.rating_matrix[user_idx].toarray().flatten()
        rated_movies = np.where(user_ratings > 0)[0]
        
        # Compute similarities once
        user_similarities = self.compute_user_similarity(user_idx)
        
        # Predict ratings for all unrated movies
        predictions = []
        
        for movie_idx in range(self.rating_matrix.shape[1]):
            # Skip already rated movies
            if exclude_rated and movie_idx in rated_movies:
                continue
            
            movie_id = self.reverse_movie_map[movie_idx]
            
            # Predict rating
            pred_rating = self.predict_rating(user_id, movie_id)
            predictions.append((movie_id, pred_rating))
        
        # Sort by predicted rating (descending)
        predictions.sort(key=lambda x: x[1], reverse=True)
        
        return predictions[:n_recommendations]


In [ ]:
# Initialize and fit the User–User Collaborative Filtering model
uu_model = UserUserCollaborativeFiltering(top_k_neighbors=50)

print("\n Model initialized!")
print("Top-K neighbors:", uu_model.top_k)

print("\n Starting model fitting...")
uu_model.fit(netflix_ratings_filtered)
print("\n Model fitting complete!")


In [ ]:
# Pick one known user–movie pair from the training dataset
user_id = netflix_ratings_filtered['UserID'].iloc[0]
movie_id = netflix_ratings_filtered['MovieID'].iloc[1]

print(f"\n Predicting score for User {user_id} and Movie {movie_id} ...")
predicted = uu_model.predict_rating(user_id, movie_id)

print(f" Predicted Rating (User {user_id}, Movie {movie_id}): {predicted:.2f}")


In [ ]:
import os
import pickle

# Create a directory to store saved models
os.makedirs("saved_models", exist_ok=True)

# Save metadata necessary to rebuild the model
uu_meta = {
    "user_id_map": uu_model.user_id_map,
    "movie_id_map": uu_model.movie_id_map,
    "reverse_user_map": uu_model.reverse_user_map,
    "reverse_movie_map": uu_model.reverse_movie_map,
    "user_means": uu_model.user_means,
    "global_mean": uu_model.global_mean
}

with open("saved_models/user_user_meta.pkl", "wb") as f:
    pickle.dump(uu_meta, f)

print(" User–User CF model saved successfully to 'saved_models/user_user_meta.pkl'")


In [ ]:
import pickle

# Recreate an empty instance
uu_model = UserUserCollaborativeFiltering(top_k_neighbors=50)

# Load mappings and stats from the saved file
with open("saved_models/user_user_meta.pkl", "rb") as f:
    uu_meta = pickle.load(f)

uu_model.user_id_map = uu_meta["user_id_map"]
uu_model.movie_id_map = uu_meta["movie_id_map"]
uu_model.reverse_user_map = uu_meta["reverse_user_map"]
uu_model.reverse_movie_map = uu_meta["reverse_movie_map"]
uu_model.user_means = uu_meta["user_means"]
uu_model.global_mean = uu_meta["global_mean"]

print(" User–User CF model reloaded successfully and ready for evaluation!")


### 5.3 User-user prediction example

### 5.4 Item-item collaborative filtering

### 5.5 Item-item rating prediction

In [ ]:
"""
Assignment 2B.2: Item-Item (Movie-Movie) Collaborative Filtering
Uses movie similarity to predict ratings
"""

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')


class ItemItemCollaborativeFiltering:
    """
    Item-Item (Movie-Movie) Collaborative Filtering Recommender System.
    """

    def __init__(self, top_k_neighbors=50):
        self.top_k = top_k_neighbors
        self.rating_matrix = None
        self.movie_similarity_matrix = None
        self.user_id_map = {}
        self.movie_id_map = {}
        self.reverse_user_map = {}
        self.reverse_movie_map = {}
        self.global_mean = None

    # ----------------------------------------------------------------------
    def create_rating_matrix(self, ratings_df):
        """Create user-item rating matrix from ratings dataframe."""
        print("Creating rating matrix...")

        unique_users = ratings_df['UserID'].unique()
        unique_movies = ratings_df['MovieID'].unique()

        self.user_id_map = {uid: idx for idx, uid in enumerate(unique_users)}
        self.movie_id_map = {mid: idx for idx, mid in enumerate(unique_movies)}
        self.reverse_user_map = {idx: uid for uid, idx in self.user_id_map.items()}
        self.reverse_movie_map = {idx: mid for mid, idx in self.movie_id_map.items()}

        row_indices = [self.user_id_map[uid] for uid in ratings_df['UserID']]
        col_indices = [self.movie_id_map[mid] for mid in ratings_df['MovieID']]
        ratings_data = ratings_df['Rating'].values

        self.rating_matrix = csr_matrix(
            (ratings_data, (row_indices, col_indices)),
            shape=(len(unique_users), len(unique_movies))
        )

        print(f" Rating matrix created: {self.rating_matrix.shape}")
        print(f"   Users: {len(unique_users):,}")
        print(f"   Movies: {len(unique_movies):,}")
        print(f"   Ratings: {len(ratings_data):,}")
        print(f"   Sparsity: {100 * (1 - len(ratings_data) / (len(unique_users) * len(unique_movies))):.2f}%")
        return self.rating_matrix

    # ----------------------------------------------------------------------
    def compute_movie_similarity_matrix(self):
        """Compute movie–movie cosine similarity matrix."""
        print("\nComputing movie–movie similarity matrix...")
        print("Warning:  This may take several minutes for large datasets!")

        movie_matrix = self.rating_matrix.T
        self.movie_similarity_matrix = cosine_similarity(movie_matrix, dense_output=False)

        if self.movie_similarity_matrix.shape[0] < 20000:
            self.movie_similarity_matrix = self.movie_similarity_matrix.toarray()

        print(f" Movie similarity matrix computed: {self.movie_similarity_matrix.shape}")
        return self.movie_similarity_matrix

    # ----------------------------------------------------------------------
    def fit(self, ratings_df):
        """Fit the Item–Item CF model on training data."""
        print("\n" + "="*80)
        print("FITTING ITEM–ITEM COLLABORATIVE FILTERING MODEL")
        print("="*80)

        self.create_rating_matrix(ratings_df)
        self.global_mean = ratings_df['Rating'].mean()
        print(f"\n Global mean rating: {self.global_mean:.4f}")
        self.compute_movie_similarity_matrix()
        print(f"\n Model fitted successfully!")
        print("="*80 + "\n")

    # ----------------------------------------------------------------------
    def predict_rating(self, user_id, movie_id):
        """
        Predict rating for a user–movie pair using Item–Item CF.
        """

        # Warning: Safety check: ensure rating matrix exists
        if self.rating_matrix is None:
            raise ValueError("Rating matrix not initialized. Call create_rating_matrix() before predictions.")

        if user_id not in self.user_id_map or movie_id not in self.movie_id_map:
            return self.global_mean

        user_idx = self.user_id_map[user_id]
        movie_idx = self.movie_id_map[movie_id]

        user_ratings = self.rating_matrix[user_idx].toarray().flatten()
        rated_movies = np.where(user_ratings > 0)[0]
        if len(rated_movies) == 0:
            return self.global_mean

        if isinstance(self.movie_similarity_matrix, np.ndarray):
            movie_similarities = self.movie_similarity_matrix[movie_idx, rated_movies]
        else:
            movie_similarities = self.movie_similarity_matrix[movie_idx, rated_movies].toarray().flatten()

        ratings_for_similar = user_ratings[rated_movies]
        positive_mask = movie_similarities > 0
        if np.sum(positive_mask) == 0:
            return self.global_mean

        movie_similarities = movie_similarities[positive_mask]
        ratings_for_similar = ratings_for_similar[positive_mask]

        if len(movie_similarities) > self.top_k:
            top_k_indices = np.argsort(movie_similarities)[-self.top_k:]
            movie_similarities = movie_similarities[top_k_indices]
            ratings_for_similar = ratings_for_similar[top_k_indices]

        numerator = np.sum(movie_similarities * ratings_for_similar)
        denominator = np.sum(np.abs(movie_similarities))
        if denominator == 0:
            return self.global_mean

        predicted_rating = np.clip(numerator / denominator, 1, 5)
        return predicted_rating

    # ----------------------------------------------------------------------
    def recommend_for_user(self, user_id, n_recommendations=10, exclude_rated=True):
        """Generate top-N recommendations for a given user."""
        if user_id not in self.user_id_map:
            return []

        user_idx = self.user_id_map[user_id]
        user_ratings = self.rating_matrix[user_idx].toarray().flatten()
        rated_movies = np.where(user_ratings > 0)[0]

        predictions = []
        for movie_idx in range(self.rating_matrix.shape[1]):
            if exclude_rated and movie_idx in rated_movies:
                continue
            movie_id = self.reverse_movie_map[movie_idx]
            pred_rating = self.predict_rating(user_id, movie_id)
            predictions.append((movie_id, pred_rating))

        predictions.sort(key=lambda x: x[1], reverse=True)
        return predictions[:n_recommendations]

    # ----------------------------------------------------------------------
    def get_similar_movies(self, movie_id, n_similar=10):
        """Get the most similar movies to a given movie."""
        if movie_id not in self.movie_id_map:
            return []

        movie_idx = self.movie_id_map[movie_id]
        if isinstance(self.movie_similarity_matrix, np.ndarray):
            similarities = self.movie_similarity_matrix[movie_idx]
        else:
            similarities = self.movie_similarity_matrix[movie_idx].toarray().flatten()

        similar_indices = np.argsort(similarities)[::-1][1:n_similar+1]
        return [(self.reverse_movie_map[idx], similarities[idx]) for idx in similar_indices]

    # ----------------------------------------------------------------------
    def save_similarity_matrix(self, filepath='movie_similarity_matrix.npy'):
        """Save precomputed movie–movie similarity matrix."""
        np.save(filepath, self.movie_similarity_matrix)
        print(f" Similarity matrix saved to {filepath}")

    def load_similarity_matrix(self, filepath='movie_similarity_matrix.npy'):
        """Load precomputed similarity matrix."""
        self.movie_similarity_matrix = np.load(filepath)
        print(f" Similarity matrix loaded from {filepath}")

    # ----------------------------------------------------------------------
    def restore_after_load(self, ratings_df):
        """
        Restore full model after loading similarity matrix + metadata.
        Automatically rebuilds rating matrix and global mean.
        """
        if self.movie_similarity_matrix is None:
            raise ValueError("Warning: Load similarity matrix before restoring model.")
        print("🔧 Rebuilding rating matrix from dataset...")
        self.create_rating_matrix(ratings_df)
        self.global_mean = ratings_df["Rating"].mean()
        print(f" Model fully restored. Global mean: {self.global_mean:.4f}")


In [ ]:
# Initialize and fit the Item–Item Collaborative Filtering model
ii_model = ItemItemCollaborativeFiltering(top_k_neighbors=50)

print("\n Model initialized!")
print("Top-K neighbors:", ii_model.top_k)

print("\n Starting model fitting...")
ii_model.fit(netflix_ratings_filtered)
print("\n Model fitting complete!")


In [ ]:
# Quick sanity check after restore
sample_user = list(ii_model.user_id_map.keys())[0]
sample_movie = list(ii_model.movie_id_map.keys())[0]

pred = ii_model.predict_rating(sample_user, sample_movie)
print(f"Predicted rating for user {sample_user}, movie {sample_movie}: {pred:.4f}")

# Check that it's not just global mean
print("Global mean:", ii_model.global_mean)
if abs(pred - ii_model.global_mean) < 1e-6:
    print("Warning: Warning: Predictions might still default to mean (check rating_matrix rebuild).")
else:
    print(" Model is predicting personalized scores!")


### 5.6 Example item-item prediction

In [ ]:
import random

# Pick a random user and movie from the dataset
user_id = random.choice(netflix_ratings_filtered['UserID'].unique())
movie_id = random.choice(netflix_ratings_filtered['MovieID'].unique())

print(f"\n Predicting score for User {user_id} and Movie {movie_id} ...")
predicted = ii_model.predict_rating(user_id, movie_id)

print(f" Predicted Rating (User {user_id}, Movie {movie_id}): {predicted:.2f}")


In [ ]:
import os
import pickle
import numpy as np

# Create folder for saved models
os.makedirs("saved_models", exist_ok=True)

# Save similarity matrix for reuse
ii_model.save_similarity_matrix("saved_models/item_similarity_matrix.npy")

# Save essential mappings and metadata
ii_meta = {
    "user_id_map": ii_model.user_id_map,
    "movie_id_map": ii_model.movie_id_map,
    "reverse_user_map": ii_model.reverse_user_map,
    "reverse_movie_map": ii_model.reverse_movie_map,
    "global_mean": ii_model.global_mean
}

with open("saved_models/item_item_meta.pkl", "wb") as f:
    pickle.dump(ii_meta, f)

print(" Item–Item CF model saved successfully!")
print("   → saved_models/item_similarity_matrix.npy")
print("   → saved_models/item_item_meta.pkl")


In [ ]:
import pickle
import numpy as np

# Recreate empty instance
ii_model = ItemItemCollaborativeFiltering(top_k_neighbors=50)

# Load similarity matrix
ii_model.load_similarity_matrix("saved_models/item_similarity_matrix.npy")

# Load metadata
with open("saved_models/item_item_meta.pkl", "rb") as f:
    ii_meta = pickle.load(f)

ii_model.user_id_map = ii_meta["user_id_map"]
ii_model.movie_id_map = ii_meta["movie_id_map"]
ii_model.reverse_user_map = ii_meta["reverse_user_map"]
ii_model.reverse_movie_map = ii_meta["reverse_movie_map"]
ii_model.global_mean = ii_meta["global_mean"]

print(" Item–Item CF model reloaded successfully and ready for evaluation!")


### 5.7 Sample recommendations for a selected user

In [ ]:
import random

# ---------------------------------------------------------------
#  Select a random user from the training dataset
# ---------------------------------------------------------------
user_id = random.choice(netflix_ratings_filtered["UserID"].unique())
print(f"\n Generating recommendations for User {user_id} ...")

# ---------------------------------------------------------------
#  Collect all movies this user has rated
# ---------------------------------------------------------------
user_movies = netflix_ratings_filtered.loc[
    netflix_ratings_filtered["UserID"] == user_id, "MovieID"
].unique()

if len(user_movies) == 0:
    raise ValueError(f"User {user_id} has no rated movies in the dataset.")

# If fewer than 5 movies rated, use all; otherwise, sample 5 randomly
sample_movies = random.sample(list(user_movies), min(5, len(user_movies)))

print(f"\n Sample of {len(sample_movies)} rated movies for User {user_id}: {sample_movies}\n")

# ---------------------------------------------------------------
#  For each of the 5 movies, show top 5 similar movies
# ---------------------------------------------------------------
for i, movie_id in enumerate(sample_movies, start=1):
    similar_movies = ii_model.get_similar_movies(movie_id=movie_id, n_similar=5)

    print(f" {i}. Movie {movie_id} → Top 5 Similar Movies:")
    if not similar_movies:
        print("   Warning: No similar movies found (for this movie ID).")
    else:
        for rank, (sim_movie_id, sim_score) in enumerate(similar_movies, start=1):
            print(f"   {rank:2d}. Movie {sim_movie_id}  (Similarity = {sim_score:.4f})")
    print("-" * 60)


### 5.8 Evaluate item-item collaborative filtering with RMSE

#### Item-item RMSE

In [ ]:
from sklearn.metrics import mean_squared_error
from tqdm import tqdm
import numpy as np
import math

print("\n Evaluating Item–Item CF model on test data (batched)...")

# --- Prepare dataset ---
probe_filtered = probe_filtered.dropna(subset=["UserID", "MovieID", "Rating"])
print(f" Cleaned test dataset: {len(probe_filtered):,} rows remain")

# --- Batch setup ---
BATCH_SIZE = 5000
num_batches = math.ceil(len(probe_filtered) / BATCH_SIZE)
print(f" Processing in {num_batches} batches of {BATCH_SIZE} samples each...\n")

# --- Storage ---
ii_true_all, ii_pred_all = [], []
batch_rmse_values = []


In [ ]:
for b in range(num_batches):
    start_idx = b * BATCH_SIZE
    end_idx = min((b + 1) * BATCH_SIZE, len(probe_filtered))
    batch = probe_filtered.iloc[start_idx:end_idx]

    y_true_batch, y_pred_batch = [], []

    print(f"  Batch {b+1}/{num_batches} → Rows {start_idx:,} to {end_idx-1:,}")

    for _, row in tqdm(batch.iterrows(), total=len(batch), leave=False):
        try:
            pred = ii_model.predict_rating(row["UserID"], row["MovieID"])
            pred = float(pred)
            if np.isnan(pred) or np.isinf(pred):
                pred = ii_model.global_mean
        except Exception:
            pred = ii_model.global_mean

        y_true_batch.append(float(row["Rating"]))
        y_pred_batch.append(pred)

    # --- Filter invalid values ---
    y_true_batch = np.array(y_true_batch)
    y_pred_batch = np.array(y_pred_batch)
    valid_mask = np.isfinite(y_pred_batch) & np.isfinite(y_true_batch)
    y_true_batch = y_true_batch[valid_mask]
    y_pred_batch = y_pred_batch[valid_mask]

    # --- Batch RMSE ---
    if len(y_true_batch) > 0:
        batch_rmse = np.sqrt(mean_squared_error(y_true_batch, y_pred_batch))
        batch_rmse_values.append(batch_rmse)
        print(f"    Batch {b+1} RMSE: {batch_rmse:.4f}")
    else:
        print(f"   Warning: Batch {b+1} skipped (invalid predictions only)")

    ii_true_all.extend(y_true_batch)
    ii_pred_all.extend(y_pred_batch)
    print("-" * 60)


In [ ]:
final_rmse = np.sqrt(mean_squared_error(ii_true_all, ii_pred_all))

print("\n# ----------------------------------------------------------------------")
print(f" Final Item–Item CF RMSE: {final_rmse:.4f}")
print("# ----------------------------------------------------------------------")

# Also summarize batch RMSEs
if batch_rmse_values:
    print(f" Average batch RMSE: {np.mean(batch_rmse_values):.4f}")
    print(f" Min batch RMSE: {np.min(batch_rmse_values):.4f}")
    print(f" Max batch RMSE: {np.max(batch_rmse_values):.4f}")


In [ ]:
import json, os

os.makedirs("saved_models", exist_ok=True)

results = {
    "Item-Item CF Final RMSE": float(final_rmse),
    "Average Batch RMSE": float(np.mean(batch_rmse_values)),
    "Min Batch RMSE": float(np.min(batch_rmse_values)),
    "Max Batch RMSE": float(np.max(batch_rmse_values)),
}

with open("saved_models/item_item_rmse.json", "w") as f:
    json.dump(results, f, indent=4)

print(" Saved Final Item–Item CF RMSE results to saved_models/item_item_rmse.json")


### 5.9 Evaluate user-user collaborative filtering with RMSE

#### User-user RMSE

In [ ]:
from sklearn.metrics import mean_squared_error
from tqdm import tqdm
import numpy as np
import math

print("\n Evaluating User–User CF model on test data (batched)...")

# --- Prepare dataset ---
probe_filtered = probe_filtered.dropna(subset=["UserID", "MovieID", "Rating"])
print(f" Cleaned test dataset: {len(probe_filtered):,} rows remain")

# --- Batch setup ---
BATCH_SIZE = 5000
num_batches = math.ceil(len(probe_filtered) / BATCH_SIZE)
print(f" Processing in {num_batches} batches of {BATCH_SIZE} samples each...\n")

# --- Storage ---
uu_true_all, uu_pred_all = [], []

for b in range(num_batches):
    start_idx = b * BATCH_SIZE
    end_idx = min((b + 1) * BATCH_SIZE, len(probe_filtered))
    batch = probe_filtered.iloc[start_idx:end_idx]

    y_true_batch, y_pred_batch = [], []

    print(f"  Batch {b+1}/{num_batches} → Rows {start_idx:,} to {end_idx-1:,}")

    for _, row in tqdm(batch.iterrows(), total=len(batch), leave=False):
        try:
            pred = uu_model.predict_rating(row["UserID"], row["MovieID"])
            pred = float(pred)
            if np.isnan(pred) or np.isinf(pred):
                pred = uu_model.global_mean
        except Exception:
            pred = uu_model.global_mean

        y_true_batch.append(float(row["Rating"]))
        y_pred_batch.append(pred)

    # --- Filter invalid values ---
    y_true_batch = np.array(y_true_batch)
    y_pred_batch = np.array(y_pred_batch)
    valid_mask = np.isfinite(y_pred_batch) & np.isfinite(y_true_batch)
    y_true_batch = y_true_batch[valid_mask]
    y_pred_batch = y_pred_batch[valid_mask]

    # --- Batch RMSE ---
    if len(y_true_batch) > 0:
        batch_rmse = np.sqrt(mean_squared_error(y_true_batch, y_pred_batch))
        print(f"    Batch {b+1} RMSE: {batch_rmse:.4f}")
    else:
        print(f"   Warning: Batch {b+1} skipped (invalid predictions only)")

    uu_true_all.extend(y_true_batch)
    uu_pred_all.extend(y_pred_batch)
    print("-" * 60)


In [ ]:
final_rmse_user = np.sqrt(mean_squared_error(uu_true_all, uu_pred_all))
print("\n# ----------------------------------------------------------------------")
print(f" Final USER–USER CF RMSE (Subset Model): {final_rmse_user:.4f}")
print("# ----------------------------------------------------------------------")


### 5.10 Compare user-user and item-item RMSE

In [ ]:
import json
import matplotlib.pyplot as plt

# Load saved RMSEs
with open("saved_models/user_user_rmse.json") as f:
    user_rmse = list(json.load(f).values())[0]
with open("saved_models/item_item_rmse.json") as f:
    item_rmse = list(json.load(f).values())[0]

models = ["User–User CF", "Item–Item CF"]
values = [user_rmse, item_rmse]

plt.figure(figsize=(7,5))
bars = plt.bar(models, values, color=["skyblue", "lightgreen"])
plt.title("Comparison of Collaborative Filtering RMSEs", fontsize=14)
plt.ylabel("RMSE (Lower = Better)", fontsize=12)
plt.ylim(0, max(values) + 0.1)

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 0.02, f"{height:.4f}", ha="center", fontsize=11, fontweight="bold")

# Add difference annotation
diff = abs(user_rmse - item_rmse)
better = "User–User" if user_rmse < item_rmse else "Item–Item"
plt.text(0.5, max(values) - 0.05, f" {better} CF performs better by {diff:.4f} RMSE", 
         ha="center", fontsize=11, color="darkgreen")

plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.show()
